# Super-resolution trained for handwriting: does it help PaddleOCR read?

This notebook holds two experiments:

| | What | Status |
|---|---|---|
| **Experiment 2** (this part, first) | a new **handwriting line upscaler**: any line width, random x2/x3/x4 damage, stroke-preserving loss, judged by PaddleOCR | new |
| **Experiment 1** (further down) | ESPCN x4 fine-tuned on IAM / our photos | done, results kept below |

Experiment 1 found that higher PSNR did **not** mean better reading (`espcn-iam`: best PSNR, worst CER). So Experiment 2 does **not** assume super-resolution helps: PaddleOCR's CER on held-out lines is the main judge, and the best checkpoint is picked by **validation CER**, not PSNR.

**How to run Experiment 2 only:** run the install cell (0a), then every cell down to the heading *Experiment 1*. The first cell of Experiment 1 stops "Run All" on purpose so the old experiment is not re-run by accident.

## 0a. Install the packages this notebook needs
Installs only what is missing (here most likely **PyTorch**), into the kernel's own Python, at the versions pinned in `requirements.txt` when pinned. Safe to re-run.

In [15]:
import importlib
import importlib.util
import re
import subprocess
import sys
from pathlib import Path

NEEDED = {
    "torch": "torch",                   # training the CNN
    "cv2": "opencv-contrib-python",     # includes cv2.dnn_superres (pretrained ESPCN)
    "numpy": "numpy",
    "pandas": "pandas",
    "openpyxl": "openpyxl",             # reading the .xlsx labels
    "matplotlib": "matplotlib",
    "jiwer": "jiwer",
    "yaml": "PyYAML",                   # augmentation policy
    "PIL": "pillow",
    "paddle": "paddlepaddle",
    "paddleocr": "paddleocr",
}

req_file = next((p / "requirements.txt" for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()), None)
pins = {}
if req_file:
    for line in req_file.read_text().splitlines():
        m = re.match(r"\s*([A-Za-z0-9_.\-]+)\s*==\s*([^\s#]+)", line)
        if m:
            pins[m.group(1).lower()] = m.group(2)


def installed(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return module in sys.modules


def has_dnn_superres():
    code = "import cv2; cv2.dnn_superres.DnnSuperResImpl_create()"
    return subprocess.run([sys.executable, "-c", code], capture_output=True).returncode == 0


missing = [pkg for mod, pkg in NEEDED.items() if not installed(mod)]
if "opencv-contrib-python" not in missing and not has_dnn_superres():
    missing.append("opencv-contrib-python")
to_install = [f"{p}=={pins[p.lower()]}" if p.lower() in pins else p for p in missing]

print("kernel Python:", sys.executable)
if to_install:
    print("installing:", " ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install])
    importlib.invalidate_caches()
    print("done. If an import below still fails: Kernel > Restart, then Run All.")
else:
    print("all required packages are already installed")

kernel Python: <LOCAL_ASSIGNMENT>/.venv-paddle/bin/python
all required packages are already installed


# Experiment 2: a handwriting line upscaler

**Idea.** Take clear handwritten lines (sharp target, *HR*). Damage them on purpose the way a far-away phone photo is damaged (smaller, blurry, a bit of shake, noise, JPEG, lighting, glare) to get the small input (*LR*). Train a small CNN to undo the damage. Then ask PaddleOCR to read: no enlargement vs bicubic vs pretrained SR vs our model, on lines from boards the model never saw.

**What is different from Experiment 1**
- **One model for x2, x3 and x4**, for **any line width**. The line is first enlarged with bicubic to the target height, and the CNN adds a correction (*residual learning*). Its last layer starts at zero, so before training it is **exactly bicubic**: training can only move it away from bicubic if that lowers the loss.
- **Whole lines, aspect ratio kept**: every line is resized to `HR_HEIGHT` px tall; its width follows.
- **Richer damage**: random x2/x3/x4, blur, mild motion blur, JPEG, noise, brightness/contrast, mild glare. No perspective distortion (the pipeline corrects that earlier).
- **Loss** = Charbonnier (a smooth L1) + an **edge loss** that compares stroke edges (Sobel gradients), so stroke boundaries count, not only average brightness. No adversarial (GAN) loss: GANs invent detail, i.e. wrong letters.
- **Best epoch = lowest validation CER** (PaddleOCR reads the validation lines after every epoch).

**Rule decided before training** (on held-out *whiteboard* lines): the model "helps" only if, at x3 and x4 together, its CER is lower than **both** `none` and `bicubic` by at least `GAIN_NEEDED_SR` with a 95% confidence interval above 0, and at x2 it is not worse than `none` by more than 0.01.

In [16]:
# ============================ Experiment 2 settings (change these) ============================
# ---- data ----
HR_HEIGHT = 64            # px: every clear line is resized to this height (aspect ratio kept) = the model's output height
MIN_SOURCE_H = 64         # px: only lines at least this tall in the photo can be sharp targets (a target is never enlarged)
MAX_LABEL_CER = 0.15      # a photo line counts as "clear" only if PaddleOCR already reads it (almost) right at full size
USE_IAM = True            # also use IAM handwriting lines (many writers, pen on paper)
IAM_LINES = {"train": 1500, "val": 150, "test": 100}   # IAM lines per split (IAM's own train / val / test lists)
SPLIT_BY = "board"        # "board": every photo of one whiteboard (= one writer, one session) stays in ONE split (no leak)
                          # "photo": every photo stays in one split, but other photos of the SAME board may be in train
MANUAL_SPLIT = None       # e.g. {"lecture": "train", "cnn": "test", ...} to choose the boards yourself (board mode)
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
MAX_SAME_TEXT_EVAL = 5    # val/test: at most this many crops (photos) of the same line text per board (stops one line dominating)

# ---- how the small input (LR) is made from a sharp line (HR) ----
SCALES = (2, 3, 4)                                  # shrink factor: a 64 px line becomes 32 / 21 / 16 px tall
BLUR_SIGMA = (0.3, 0.8)                             # Gaussian blur (focus / lens), in small-image pixels
MOTION_BLUR_PROB, MOTION_BLUR_LEN = 0.3, (1.0, 2.5) # mild hand shake: how often, and length in small-image pixels
NOISE_SIGMA = (0.0, 6.0)                            # sensor noise (grey levels)
JPEG_QUALITY = (40, 90)                             # phone compression
BRIGHTNESS, CONTRAST = (-25, 25), (0.75, 1.15)      # lighting
GLARE_PROB, GLARE_STRENGTH = 0.25, (0.15, 0.45)     # soft bright spot that washes out part of the line
# (no perspective distortion: the pipeline corrects perspective before this step)

# ---- model ----
CHANNELS = 32             # feature maps per layer
RES_BLOCKS = 6            # residual blocks; more = sees more context, slower

# ---- training ----
PATCH_WIDTH = 256         # px of line (at HR_HEIGHT) per training sample; the model itself accepts any width
BATCH_SIZE = 16
STEPS_PER_EPOCH_SR = 200
MAX_EPOCHS_SR = 40
LR_START = 3e-4           # Adam learning rate at the start
LOSS = "charbonnier"      # "charbonnier" (smooth L1) or "l1"
EDGE_WEIGHT = 0.1         # weight of the edge / stroke loss (0 = off)
IAM_SHARE = 0.5           # share of each batch taken from IAM lines (the rest from our whiteboard lines)
VAL_OCR_LINES = 60        # whiteboard validation lines read by PaddleOCR after every epoch (x every scale)
LR_HALVE_AFTER, STOP_AFTER = 3, 8   # epochs without a better validation CER -> halve learning rate / stop
TRAIN_ON_GPU = True       # use the Mac GPU ("mps") for training when available
RETRAIN_SR = False        # False: reuse models/superres/handwriting_sr_best.pt if it was trained with these settings
SEED_SR = 0

# ---- evaluation ----
EVAL_METHODS = ["none", "bicubic", "bicubic+sharpen", "espcn", "fsrcnn", "realesrgan", "espcn-board", "handwriting-sr"]
# also possible: "edsr" (~40M weights: seconds per line on CPU), "espcn-iam" (Experiment 1),
# "text_gestalt", "text_telescope" (fixed 16x64 word input; see the pretrained notebook).
# "realesrgan" is the slowest method in the list: remove it for a quick run.
MAX_TEST_LINES = 200      # per source (whiteboard / IAM)
RUN_REAL_FAR_TEST = True  # also test on the REAL far-away photos of the test boards (no synthetic damage)
REAL_TEST_METHODS = ["none", "bicubic", "bicubic+sharpen", "espcn", "espcn-board", "handwriting-sr"]
GAIN_NEEDED_SR = 0.02     # success rule (see above)

In [17]:
import os
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import copy
import importlib.util
import itertools
import json
import sys
import time
from pathlib import Path

import cv2
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from paddleocr import TextDetection, TextRecognition


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "notebooks").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the whiteboard-ocr repo folder")


REPO = find_repo_root()
DATA = REPO / "data" / "board2notes_v1"
IMG_DIR = DATA / "images"
LABELS = DATA / "board2notes_labels_v1_textonly.xlsx"
IAM_DIR = REPO / "data" / "iam_line_paddle"
SR_DIR = REPO / "models" / "superres"
OUT_DIR = REPO / "outputs" / "super_resolution"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = SR_DIR / "handwriting_sr_best.pt"
CKPT_META = SR_DIR / "handwriting_sr_best.json"
REALESRGAN_WEIGHTS = SR_DIR / "RealESRGAN_x4plus.pth"
PADDLEOCR_SR_SOURCE = SR_DIR.resolve().parent / "PaddleOCR-text-sr"
TEXT_SR_WEIGHTS = {"text_gestalt": SR_DIR / "text_gestalt" / "sr_tsrn_transformer_strock_train" / "best_accuracy.pdparams",
                   "text_telescope": SR_DIR / "text_telescope" / "Telescope_train" / "best_accuracy.pdparams"}
DET_ARGS = dict(thresh=0.20, box_thresh=0.40, unclip_ratio=2.00)   # same detector settings as notebook 04
TRAIN_DEVICE = torch.device("mps" if TRAIN_ON_GPU and torch.backends.mps.is_available() else "cpu")
INFER_DEVICE = torch.device("cpu")   # speed is measured on CPU, like the OpenCV models (fair comparison)

labels = pd.read_excel(LABELS, sheet_name="Line Labels", keep_default_na=False)
images = pd.read_excel(LABELS, sheet_name="Images", keep_default_na=False)


def gt_lines(name):
    return labels.loc[labels.image_id == name, "ground_truth"].tolist()


def method_available(m):
    need = {"espcn": [SR_DIR / "ESPCN_x4.pb"], "fsrcnn": [SR_DIR / "FSRCNN_x4.pb"], "edsr": [SR_DIR / "EDSR_x4.pb"],
            "realesrgan": [REALESRGAN_WEIGHTS], "espcn-board": [SR_DIR / "espcn-board_x4.pt"],
            "espcn-iam": [SR_DIR / "espcn-iam_x4.pt"],
            "text_gestalt": [PADDLEOCR_SR_SOURCE, TEXT_SR_WEIGHTS["text_gestalt"]],
            "text_telescope": [PADDLEOCR_SR_SOURCE, TEXT_SR_WEIGHTS["text_telescope"]]}.get(m, [])
    missing = [str(p) for p in need if not p.exists()]
    packages = {"realesrgan": ["realesrgan", "basicsr", "torchvision"], "text_gestalt": ["paddle"],
                "text_telescope": ["paddle"]}.get(m, [])
    missing += [f"python package {p}" for p in packages if importlib.util.find_spec(p) is None]
    if m in ("espcn", "fsrcnn", "edsr") and not hasattr(getattr(cv2, "dnn_superres", None), "DnnSuperResImpl_create"):
        missing.append("opencv-contrib-python (cv2.dnn_superres)")
    if missing:
        print(f"  skipping {m}: missing {', '.join(missing)} (see models/README.md)")
    return not missing


print("checking evaluation methods:")
EVAL_METHODS = [m for m in EVAL_METHODS if method_available(m)]
REAL_TEST_METHODS = [m for m in REAL_TEST_METHODS if m in EVAL_METHODS]
det_model = TextDetection(model_name="PP-OCRv5_mobile_det")
rec_model = TextRecognition(model_name="PP-OCRv6_medium_rec")
print(f"train on: {TRAIN_DEVICE} | inference timed on: {INFER_DEVICE} | {len(images)} photos, {len(labels)} label rows")
print("methods:", ", ".join(EVAL_METHODS))

Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv5_mobile_det`.


checking evaluation methods:


Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


train on: mps | inference timed on: cpu | 108 photos, 2733 label rows
methods: none, bicubic, bicubic+sharpen, espcn, fsrcnn, realesrgan, espcn-board, handwriting-sr


### Helpers: crop lines, read them, score them, and the enlargement methods we compare

In [18]:
def load(name):
    img = cv2.imread(str(IMG_DIR / name))
    assert img is not None, f"cannot read {IMG_DIR / name}"
    return img


def line_height(poly):
    p = np.asarray(poly, float).reshape(-1, 2)
    if len(p) != 4:
        return float(np.ptp(p[:, 1]))
    return float((np.linalg.norm(p[0] - p[3]) + np.linalg.norm(p[1] - p[2])) / 2)


def quad_crop(img, poly):
    # cut a detected (possibly slanted) text box out straight
    p = np.float32(np.asarray(poly, float).reshape(-1, 2))
    if len(p) != 4:
        p = np.float32(cv2.boxPoints(cv2.minAreaRect(p)))
    s_, d_ = p.sum(1), np.diff(p, axis=1).ravel()
    p = np.float32([p[s_.argmin()], p[d_.argmin()], p[s_.argmax()], p[d_.argmax()]])
    w = int(round(max(np.linalg.norm(p[0] - p[1]), np.linalg.norm(p[3] - p[2]))))
    h = int(round(max(np.linalg.norm(p[0] - p[3]), np.linalg.norm(p[1] - p[2]))))
    M = cv2.getPerspectiveTransform(p, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))
    return cv2.warpPerspective(img, M, (max(w, 1), max(h, 1)), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def detect_full(img):
    res = det_model.predict(img, limit_side_len=max(img.shape[:2]), limit_type="max", **DET_ARGS)
    return [np.asarray(p, float).reshape(-1, 2) for p in res[0]["dt_polys"]]


def resize_to_height(img, h):
    w = max(1, int(round(img.shape[1] * h / img.shape[0])))
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA if img.shape[0] > h else cv2.INTER_CUBIC)


def to_y(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)[:, :, 0]


def ink_rows(gray):
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    rows = bw.mean(1) / 255
    on = np.nonzero(rows >= 0.25 * rows.max())[0]
    return (int(on[0]), int(on[-1])) if len(on) else (0, gray.shape[0] - 1)


def read(imgs):
    """PaddleOCR recogniser on a list of line images -> [(text, confidence)]."""
    if not imgs:
        return []
    return [(r["rec_text"], float(r["rec_score"])) for r in rec_model.predict(list(imgs))]


def norm_text(t):
    return " ".join(str(t).lower().split())       # case-insensitive, spaces collapsed


def cer(gt, txt):
    return min(jiwer.cer(norm_text(gt), norm_text(txt) or " "), 1.0)


def text_scores(gt, txt):
    g, t = norm_text(gt), norm_text(txt)
    return dict(cer=min(jiwer.cer(g, t or " "), 1.0),
                wer=min(jiwer.wer(g, t or "<empty>"), 1.0),           # capped at 1 like CER
                exact=float(g == t))


def psnr_y(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 10 * np.log10(255 ** 2 / max(mse, 1e-10))


def ssim_y(a, b):
    # standard SSIM (Gaussian window 11, sigma 1.5) on brightness
    a, b = a.astype(np.float64), b.astype(np.float64)
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    blur = lambda x: cv2.GaussianBlur(x, (11, 11), 1.5)
    ma, mb = blur(a), blur(b)
    va, vb, cab = blur(a * a) - ma ** 2, blur(b * b) - mb ** 2, blur(a * b) - ma * mb
    return float(np.mean(((2 * ma * mb + c1) * (2 * cab + c2)) / ((ma ** 2 + mb ** 2 + c1) * (va + vb + c2))))


def same_size_y(out, hr):
    # brightness of output and target, cut to the same width (enlarged widths can differ by a pixel)
    w = min(out.shape[1], hr.shape[1])
    return to_y(out)[:, :w], to_y(hr)[:, :w]


# ---- pretrained / earlier methods (loaded only when used) ----
_sr_cache, _cache = {}, {}


def _sr(name, k):
    if (name, k) not in _sr_cache:
        m = cv2.dnn_superres.DnnSuperResImpl_create()
        m.readModel(str(SR_DIR / f"{name.upper()}_x{k}.pb"))
        m.setModel(name.lower(), k)
        _sr_cache[(name, k)] = m
    return _sr_cache[(name, k)]


def _realesrgan():
    # same loading as super_resolution_pretrained.ipynb (BasicSR needs an old torchvision module name)
    if "realesrgan" not in _cache:
        import types
        import torchvision.transforms.functional as tvf
        compat = types.ModuleType("torchvision.transforms.functional_tensor")
        compat.rgb_to_grayscale = tvf.rgb_to_grayscale
        sys.modules.setdefault("torchvision.transforms.functional_tensor", compat)
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        _cache["realesrgan"] = RealESRGANer(scale=4, model_path=str(REALESRGAN_WEIGHTS), model=net,
                                            tile=0, tile_pad=10, pre_pad=0, half=False, gpu_id=None)
    return _cache["realesrgan"]


def _text_sr(img, size, method):
    # PaddleOCR text-SR research models: fixed 16x64 word input -> 32x128 (same as the pretrained notebook)
    import paddle
    if method not in _cache:
        sys.path.insert(0, str(PADDLEOCR_SR_SOURCE))
        from ppocr.modeling.architectures import build_model
        algo, tr = ("Gestalt", "TSRN") if method == "text_gestalt" else ("Telescope", "TBSRN")
        model = build_model(dict(model_type="sr", algorithm=algo, Transform=dict(name=tr, STN=True, infer_mode=True)))
        model.set_state_dict(paddle.load(str(TEXT_SR_WEIGHTS[method])))
        model.eval()
        _cache[method] = model
    x = cv2.resize(img, (64, 16), interpolation=cv2.INTER_AREA)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None].astype("float32") / 255.0
    with paddle.no_grad():
        y = _cache[method](paddle.to_tensor(x))["sr_img"][0].numpy().transpose(1, 2, 0)
    y = cv2.cvtColor(np.clip(y * 255, 0, 255).astype("uint8"), cv2.COLOR_RGB2BGR)
    return cv2.resize(y, size, interpolation=cv2.INTER_CUBIC)


class ESPCN(nn.Module):
    # the x4 model of Experiment 1 (only to compare with it)
    def __init__(self, r=4):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 5, padding=2)
        self.conv2 = nn.Conv2d(64, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, r * r, 3, padding=1)
        self.shuffle = nn.PixelShuffle(r)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        return torch.tanh(self.shuffle(self.conv3(x)))


def _espcn_exp1(img, method):
    if method not in _cache:
        m = ESPCN()
        m.load_state_dict(torch.load(SR_DIR / f"{method}_x4.pt", map_location="cpu"))
        _cache[method] = m.eval()
    ycc = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    y = torch.from_numpy(np.ascontiguousarray(ycc[:, :, 0], dtype=np.float32)[None, None] / 255)
    with torch.no_grad():
        out = _cache[method](y).cpu().numpy()[0, 0]
    big = cv2.resize(ycc, (img.shape[1] * 4, img.shape[0] * 4), interpolation=cv2.INTER_CUBIC)
    big[:, :, 0] = np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(big, cv2.COLOR_YCrCb2BGR)


NEW_MODEL = {}   # filled after training: {"model": the trained handwriting upscaler}


def enlarge_any(img, f, method):
    """Enlarge a line image by factor f with one of the compared methods ('none' returns it unchanged)."""
    if method == "none" or f <= 1.05:
        return img
    size = (int(round(img.shape[1] * f)), int(round(img.shape[0] * f)))
    if method == "bicubic":
        return cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
    if method == "bicubic+sharpen":            # the current pipeline's method (unsharp mask)
        up = cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
        return cv2.addWeighted(up, 1.6, cv2.GaussianBlur(up, (0, 0), sigmaX=0.8 * f), -0.6, 0)
    if method == "handwriting-sr":
        return sr_line(NEW_MODEL["model"], img, f)
    if method in ("espcn-board", "espcn-iam"):
        up, k = _espcn_exp1(img, method), 4
    elif method == "realesrgan":
        up, k = _realesrgan().enhance(img, outscale=4.0)[0], 4
    elif method in ("text_gestalt", "text_telescope"):
        return _text_sr(img, size, method)
    elif method in ("espcn", "fsrcnn", "edsr"):
        k = int(min(4, max(2, np.ceil(f))))   # smallest x2/x3/x4 model that is big enough
        up = _sr(method, k).upsample(img)
    else:
        raise ValueError(method)
    return cv2.resize(up, size, interpolation=cv2.INTER_AREA if k > f else cv2.INTER_CUBIC)


def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## E1. Clear handwritten lines (the sharp targets)

- **Our whiteboard photos (all 108)**: detect every line at full resolution, cut it out straight, and keep it as a sharp target only if
  1. it is at least `MIN_SOURCE_H` px tall (so the target is never an enlarged image), and
  2. PaddleOCR already reads it (almost) right at full size (CER <= `MAX_LABEL_CER` against the photo's ground-truth lines). Its label is that **ground-truth line** (the human annotation), not PaddleOCR's reading.
- **IAM** (if `USE_IAM`): scanned handwriting lines from many writers. IAM crops have a different amount of empty space above/below the ink than our detector's boxes, so each IAM line is cropped or padded until the ink fills the same share of the height as in our boxes.

Every line is then resized to `HR_HEIGHT` px tall, keeping its aspect ratio. The detection step takes ~10-15 min the first time and is cached (`outputs/super_resolution/handwriting_sr_lines.pkl`).

In [19]:
pool_cache = OUT_DIR / "handwriting_sr_lines.pkl"
pool_key = json.dumps(dict(HR_HEIGHT=HR_HEIGHT, MIN_SOURCE_H=MIN_SOURCE_H, MAX_LABEL_CER=MAX_LABEL_CER,
                           DET_ARGS=DET_ARGS, photos=sorted(images.image_id)), sort_keys=True)
cached = pd.read_pickle(pool_cache) if pool_cache.exists() else None
if cached is not None and cached.get("key") == pool_key:
    board_lines = cached["lines"]
    print(f"loaded {len(board_lines)} clear whiteboard lines from {pool_cache.name} (delete it to detect again)")
else:
    board_lines = []
    t_all = time.time()
    for k, row in enumerate(images.itertuples(), 1):
        gts = gt_lines(row.image_id)
        if not gts:
            continue
        img = load(row.image_id)
        crops = [quad_crop(img, p) for p in detect_full(img)]
        crops = [c for c in crops if c.shape[0] >= MIN_SOURCE_H and c.shape[1] >= 2 * c.shape[0]]
        kept = 0
        for c, (txt, conf) in zip(crops, read(crops)):
            g = min(gts, key=lambda g: cer(g, txt))
            if cer(g, txt) <= MAX_LABEL_CER:
                board_lines.append(dict(source="board", photo=row.image_id, board=row.image_id.split("_")[0],
                                        condition=row.condition, gt=g, src_h=c.shape[0],
                                        img=resize_to_height(c, HR_HEIGHT)))
                kept += 1
        if k % 10 == 0 or k == len(images):
            print(f"[{k}/{len(images)}] {row.image_id}: {kept} clear lines | total {len(board_lines)} | {time.time() - t_all:.0f} s")
    pd.to_pickle(dict(key=pool_key, lines=board_lines), pool_cache)
assert board_lines, "no clear whiteboard lines found: lower MIN_SOURCE_H or raise MAX_LABEL_CER"

# share of the box height filled by ink in OUR boxes (used to make IAM lines look like detector crops)
INK_RATIO = float(np.median([(lambda r: (r[1] - r[0] + 1) / l["img"].shape[0])(ink_rows(to_y(l["img"])))
                             for l in board_lines]))


def iam_lines(split, n, seed):
    if not USE_IAM or n <= 0:
        return []
    fname = {"train": "train.txt", "val": "val.txt", "test": "test.txt"}[split]
    entries = [l.split("\t", 1) for l in (IAM_DIR / fname).read_text(encoding="utf-8").splitlines() if "\t" in l]
    pick = np.random.default_rng(seed).choice(len(entries), min(n, len(entries)), replace=False)
    out = []
    for i in pick:
        path, txt = entries[i]
        g = cv2.imread(str(IAM_DIR / path), cv2.IMREAD_GRAYSCALE)
        if g is None or not txt.strip():
            continue
        top, bottom = ink_rows(g)
        box = int(round((bottom - top + 1) / INK_RATIO))
        if box >= g.shape[0]:
            extra = box - g.shape[0]
            g = cv2.copyMakeBorder(g, extra // 2, extra - extra // 2, 0, 0, cv2.BORDER_CONSTANT, value=int(np.median(g)))
        else:
            y0 = int(np.clip((top + bottom) / 2 - box / 2, 0, g.shape[0] - box))
            g = g[y0:y0 + box]
        out.append(dict(source="iam", photo=f"iam_{path}", board="iam", condition="IAM scan", gt=txt.strip(),
                        src_h=g.shape[0], img=resize_to_height(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR), HR_HEIGHT),
                        split=split))
    return out


iam_pool = {s: iam_lines(s, n, SEED_SR + i) for i, (s, n) in enumerate(IAM_LINES.items())}
summary = pd.DataFrame([{k: v for k, v in l.items() if k != "img"} for l in board_lines])
print(f"{len(board_lines)} clear whiteboard lines from {summary.photo.nunique()} photos; ink = {INK_RATIO:.0%} of the box height")
display(summary.groupby("board").agg(lines=("gt", "size"), photos=("photo", "nunique"), texts=("gt", "nunique"),
                                     median_source_h=("src_h", "median")))
print("IAM lines:", {s: len(v) for s, v in iam_pool.items()})

Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizin

[10/108] television_07.png: 0 clear lines | total 3 | 57 s


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.


[20/108] cnn_04.png: 0 clear lines | total 4 | 117 s


Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.


[30/108] cnn_14.png: 0 clear lines | total 4 | 178 s


... (14 outputs left out)

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[108/108] lecture_13.png: 18 clear lines | total 269 | 682 s
269 clear whiteboard lines from 34 photos; ink = 47% of the box height


,lines,photos,texts,median_source_h
board,,,,
animal,1,1,1,193.0
cnn,18,5,6,76.5
lecture,212,13,64,81.0
quiz,35,13,5,92.0
television,3,2,2,334.0


IAM lines: {'train': 1500, 'val': 150, 'test': 100}


## E2. Train / validation / test split (no leaks)

Each board was photographed up to 49 times, and all photos of one board show **the same ink strokes by the same writer**. If photos of one board were in both training and test, the model could memorise those exact strokes and the test would look better than it should. So with `SPLIT_BY = "board"` (default) every board goes entirely into one split; the program tries every assignment of the boards and keeps the one closest to 70 / 15 / 15 (by number of lines). With only 5 boards the result is lumpy: the counts are printed so you can see it.

`SPLIT_BY = "photo"` keeps each photo in one split (more even, 70/15/15), but **leaks the writer and the strokes** across splits: use it only to see how much the leak inflates the result.

IAM uses its own train / val / test lists (they are the standard writer-independent IAM line split; the dataset card does not list writers, so this is assumed from the split sizes 6482 / 976 / 2915).

In [20]:
def assign_groups(sizes, fractions, seed):
    names, splits, total = sorted(sizes), list(fractions), sum(sizes.values())
    assert len(names) >= len(splits), f"only {len(names)} groups ({names}): cannot fill {len(splits)} splits"
    if len(names) <= 8:      # few groups (boards): try every assignment, keep the one closest to the target shares
        best = None
        for combo in itertools.product(splits, repeat=len(names)):
            if set(combo) != set(splits):
                continue
            share = {s: sum(sizes[n] for n, c in zip(names, combo) if c == s) / total for s in splits}
            err = sum((share[s] - fractions[s]) ** 2 for s in splits)
            if best is None or err < best[0] - 1e-12:
                best = (err, dict(zip(names, combo)))
        return best[1]
    # many groups (photos): shuffle, then fill train, val, test in turn
    order = list(np.random.default_rng(seed).permutation(names))
    out, done, i = {}, 0, 0
    for s in splits:
        goal = total * sum(fractions[x] for x in splits[:splits.index(s) + 1])
        while i < len(order) and (done < goal or s == splits[-1]):
            out[order[i]] = s
            done += sizes[order[i]]
            i += 1
    return out


group_key = "board" if SPLIT_BY == "board" else "photo"
sizes = summary.groupby(group_key).size().to_dict()
assignment = dict(MANUAL_SPLIT) if MANUAL_SPLIT else assign_groups(sizes, SPLIT_FRACTIONS, SEED_SR)
missing = set(sizes) - set(assignment)
assert not missing, f"MANUAL_SPLIT does not say where these go: {missing}"
for l in board_lines:
    l["split"] = assignment[l[group_key]]

# val / test: at most MAX_SAME_TEXT_EVAL crops of the same text per board (one line photographed 40x would dominate)
rng_cap = np.random.default_rng(SEED_SR)
lines_by_split = {"train": [], "val": [], "test": []}
seen = {}
for i in rng_cap.permutation(len(board_lines)):
    l = board_lines[i]
    key = (l["split"], l["board"], norm_text(l["gt"]))
    if l["split"] != "train" and seen.get(key, 0) >= MAX_SAME_TEXT_EVAL:
        continue
    seen[key] = seen.get(key, 0) + 1
    lines_by_split[l["split"]].append(l)
for s in lines_by_split:
    lines_by_split[s] += iam_pool.get(s, [])

# leak checks
for a, b in itertools.combinations(lines_by_split, 2):
    pa = {l["photo"] for l in lines_by_split[a]}; pb = {l["photo"] for l in lines_by_split[b]}
    assert not pa & pb, f"photo in both {a} and {b}: {sorted(pa & pb)[:3]}"
    if SPLIT_BY == "board":
        ba = {l["board"] for l in lines_by_split[a] if l["source"] == "board"}
        bb = {l["board"] for l in lines_by_split[b] if l["source"] == "board"}
        assert not ba & bb, f"board in both {a} and {b}: {ba & bb}"
train_texts = {norm_text(l["gt"]) for l in lines_by_split["train"] if l["source"] == "board"}
test_texts = {norm_text(l["gt"]) for l in lines_by_split["test"] if l["source"] == "board"}

rows = []
for s, ls in lines_by_split.items():
    for src in ("board", "iam"):
        part = [l for l in ls if l["source"] == src]
        rows.append(dict(split=s, source=src, lines=len(part), photos_or_files=len({l["photo"] for l in part}),
                         boards=", ".join(sorted({l["board"] for l in part})) if src == "board" else "-",
                         different_texts=len({norm_text(l["gt"]) for l in part})))
split_table = pd.DataFrame(rows)
print(f"split by {SPLIT_BY}:", assignment if SPLIT_BY == "board" else f"{len(assignment)} photos assigned")
n_board = split_table[split_table.source == "board"].set_index("split").lines
print("whiteboard lines per split (share):", {s: f"{n} ({n / max(n_board.sum(), 1):.0%})" for s, n in n_board.items()})
print(f"line texts in both train and test (whiteboard): {len(train_texts & test_texts)}")
display(split_table)
split_table.to_csv(OUT_DIR / "handwriting_sr_split.csv", index=False)
assert lines_by_split["train"] and lines_by_split["test"], "a split is empty: change SPLIT_BY / MANUAL_SPLIT"

split by board: {'animal': 'val', 'cnn': 'val', 'lecture': 'train', 'quiz': 'test', 'television': 'val'}
whiteboard lines per split (share): {'train': '212 (82%)', 'val': '22 (9%)', 'test': '24 (9%)'}
line texts in both train and test (whiteboard): 0


,split,source,lines,photos_or_files,boards,different_texts
0,train,board,212,13,lecture,64
1,train,iam,1500,1500,-,1466
2,val,board,22,8,"animal, cnn, television",9
3,val,iam,150,150,-,150
4,test,board,24,10,quiz,5
5,test,iam,100,100,-,100


## E3. Making the small, damaged input (LR) from a sharp line (HR)

Order follows a real camera: glare on the board -> lens/focus blur -> hand shake -> fewer pixels (shrink x2/x3/x4) -> lighting -> sensor noise -> JPEG. Every training sample gets new random values, so the model never sees the same damage twice.

In [21]:
def degrade_random(hr, rng, scale=None):
    """Sharp line (BGR, HR_HEIGHT tall) -> small damaged line, like a far-away phone photo."""
    s = int(scale or rng.choice(SCALES))
    img = hr.astype(np.float32)
    H, W = img.shape[:2]
    if rng.random() < GLARE_PROB:                                    # soft bright spot
        cy, cx = rng.uniform(0, H), rng.uniform(0, W)
        ry, rx = rng.uniform(0.5, 1.5) * H, rng.uniform(0.1, 0.4) * W
        yy, xx = np.mgrid[0:H, 0:W]
        mask = np.exp(-(((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2))[..., None]
        img = img + rng.uniform(*GLARE_STRENGTH) * mask * (255 - img)
    img = cv2.GaussianBlur(img, (0, 0), sigmaX=rng.uniform(*BLUR_SIGMA) * s)          # focus blur
    if rng.random() < MOTION_BLUR_PROB:                              # mild hand shake (a short line kernel)
        n = max(2, int(round(rng.uniform(*MOTION_BLUR_LEN) * s)))
        k = np.zeros((n, n), np.float32)
        k[n // 2, :] = 1
        k = cv2.warpAffine(k, cv2.getRotationMatrix2D(((n - 1) / 2, (n - 1) / 2), rng.uniform(0, 180), 1.0), (n, n))
        img = cv2.filter2D(img, -1, k / max(k.sum(), 1e-6))
    h, w = max(8, int(round(H / s))), max(8, int(round(W / s)))
    img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)     # fewer pixels
    img = (img - 128) * rng.uniform(*CONTRAST) + 128 + rng.uniform(*BRIGHTNESS)
    img = img + rng.normal(0, rng.uniform(*NOISE_SIGMA), img.shape)
    img = np.clip(img, 0, 255).astype(np.uint8)
    q = int(rng.integers(JPEG_QUALITY[0], JPEG_QUALITY[1] + 1))
    return cv2.imdecode(cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])[1], cv2.IMREAD_COLOR)


PAIR_MARGIN = 24   # px of extra line left/right while damaging, cut off afterwards (blur/JPEG at the edges stay realistic)


def has_ink(y):
    return (y < np.median(y) - 40).mean() > 0.02


def make_pair(hr_line, rng):
    """-> (bicubic-enlarged damaged patch, sharp patch), brightness in [0, 1], both HR_HEIGHT x PATCH_WIDTH."""
    W = PATCH_WIDTH + 2 * PAIR_MARGIN
    line = hr_line
    if line.shape[1] < W:                                            # short line: pad with its background colour
        pad = W - line.shape[1]
        colour = [int(v) for v in np.median(line.reshape(-1, 3), axis=0)]
        line = cv2.copyMakeBorder(line, 0, 0, pad // 2, pad - pad // 2, cv2.BORDER_CONSTANT, value=colour)
    for _ in range(5):
        x0 = int(rng.integers(0, line.shape[1] - W + 1))
        win = line[:, x0:x0 + W]
        y_hr = to_y(win)[:, PAIR_MARGIN:PAIR_MARGIN + PATCH_WIDTH]
        if has_ink(y_hr):
            break
    lr = degrade_random(win, rng)
    bic = cv2.resize(lr, (W, HR_HEIGHT), interpolation=cv2.INTER_CUBIC)
    y_bic = to_y(bic)[:, PAIR_MARGIN:PAIR_MARGIN + PATCH_WIDTH]
    return y_bic.astype(np.float32) / 255, y_hr.astype(np.float32) / 255


def make_batch(split, n, rng):
    board = [l for l in lines_by_split[split] if l["source"] == "board"]
    iam = [l for l in lines_by_split[split] if l["source"] == "iam"]
    xs, ys = [], []
    for _ in range(n):
        pool = iam if (iam and (not board or rng.random() < IAM_SHARE)) else board
        x, y = make_pair(pool[int(rng.integers(len(pool)))]["img"], rng)
        xs.append(x); ys.append(y)
    return np.stack(xs)[:, None], np.stack(ys)[:, None]            # (n, 1, HR_HEIGHT, PATCH_WIDTH)


# preview: one sharp training line, damaged at every scale
demo_line = next(l for l in lines_by_split["train"] if l["source"] == "board")["img"][:, :4 * HR_HEIGHT * 2]
fig, axes = plt.subplots(len(SCALES) + 1, 3, figsize=(15, 1.3 * (len(SCALES) + 1)))
axes[0, 0].imshow(rgb(demo_line)); axes[0, 0].set_title(f"sharp target (HR), {HR_HEIGHT} px tall", fontsize=9)
axes[0, 1].axis("off"); axes[0, 2].axis("off")
for i, s in enumerate(SCALES, 1):
    lr = degrade_random(demo_line, np.random.default_rng(i), s)
    axes[i, 0].imshow(rgb(cv2.resize(lr, demo_line.shape[1::-1], interpolation=cv2.INTER_NEAREST)))
    axes[i, 0].set_title(f"x{s} damaged input (LR, {lr.shape[0]} px tall), shown with big pixels", fontsize=9)
    axes[i, 1].imshow(rgb(cv2.resize(lr, demo_line.shape[1::-1], interpolation=cv2.INTER_CUBIC)))
    axes[i, 1].set_title("bicubic enlargement of it", fontsize=9)
    axes[i, 2].imshow(rgb(degrade_random(demo_line, np.random.default_rng(100 + i), s)))
    axes[i, 2].set_title(f"another random x{s} damage (actual size)", fontsize=9)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

<Figure size 1500x520 with 12 Axes>

## E4. The model: bicubic + a learned correction

```
small line --bicubic--> line at HR_HEIGHT --CNN--> correction --(+)--> output
                                  \_____________________________________/
```
- **Fully convolutional** (only convolutions): it works on a line of **any width**.
- 3x3 convolutions in residual blocks; every second block uses *dilation 2* (the 3x3 filter is spread out), so the model sees a wider area (about one to two letters) without more weights.
- Works on brightness (Y) like ESPCN; colour comes from bicubic.
- The last layer starts at **zero**, so the untrained model = bicubic exactly (checked below).

In [22]:
class ResBlock(nn.Module):
    def __init__(self, ch, dilation):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=dilation, dilation=dilation)

    def forward(self, x):
        return x + self.conv2(torch.relu(self.conv1(x)))


class LineSR(nn.Module):
    def __init__(self, channels=CHANNELS, blocks=RES_BLOCKS):
        super().__init__()
        self.head = nn.Conv2d(1, channels, 3, padding=1)
        self.body = nn.ModuleList([ResBlock(channels, 1 if i % 2 == 0 else 2) for i in range(blocks)])
        self.tail = nn.Conv2d(channels, 1, 3, padding=1)
        nn.init.zeros_(self.tail.weight)       # start as "bicubic + 0"
        nn.init.zeros_(self.tail.bias)

    def forward(self, x):                      # x: bicubic-enlarged brightness, (batch, 1, H, W) in [0, 1]
        f = torch.relu(self.head(x))
        for block in self.body:
            f = block(f)
        return x + self.tail(f)


def model_device(model):
    return next(model.parameters()).device


def sr_line(model, img_bgr, f):
    """Enlarge a whole line (any width) by factor f: bicubic, then the CNN's correction on brightness."""
    size = (int(round(img_bgr.shape[1] * f)), int(round(img_bgr.shape[0] * f)))
    ycc = cv2.cvtColor(cv2.resize(img_bgr, size, interpolation=cv2.INTER_CUBIC), cv2.COLOR_BGR2YCrCb)
    y = torch.from_numpy(np.ascontiguousarray(ycc[:, :, 0], dtype=np.float32)[None, None] / 255)
    with torch.no_grad():
        out = model(y.to(model_device(model))).cpu().numpy()[0, 0]
    ycc[:, :, 0] = np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)


untrained = LineSR().eval()
n_weights = sum(p.numel() for p in untrained.parameters())
test_lr = degrade_random(lines_by_split["test"][0]["img"], np.random.default_rng(1), 4)
diff = np.abs(sr_line(untrained, test_lr, 4.0).astype(int) - enlarge_any(test_lr, 4.0, "bicubic").astype(int)).max()
print(f"LineSR: {n_weights:,} weights | untrained model vs bicubic: max difference {diff} grey levels")
assert diff <= 2, "the untrained model should equal bicubic"

LineSR: 111,585 weights | untrained model vs bicubic: max difference 1 grey levels


## E5. Training

Each step: a batch of fresh (damaged, sharp) patches -> model -> **loss** -> backprop -> Adam step.
- **Reconstruction**: Charbonnier = sqrt(error^2 + 0.001^2), averaged: like L1 (does not average away edges), but smooth at 0.
- **Edge / stroke loss**: the Sobel filter measures how fast brightness changes left-right and up-down (= stroke edges). We compare those edge maps of output and target with L1, so blurred or wobbly stroke boundaries cost extra.
- After every epoch: PSNR on fixed validation patches (secondary) and **PaddleOCR CER on fixed damaged validation lines** (main). The epoch with the lowest validation CER is saved. Epoch 0 = the untrained model = bicubic, so the saved model is bicubic unless training really beats it.

In [23]:
SOBEL_X = torch.tensor([[[[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]]]) / 4
SOBEL_Y = torch.tensor([[[[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]]]]) / 4


def recon_loss(out, target):
    if LOSS == "l1":
        return (out - target).abs().mean()
    return torch.sqrt((out - target) ** 2 + 1e-6).mean()             # Charbonnier, eps = 0.001


def edge_loss(out, target):
    kx, ky = SOBEL_X.to(out.device), SOBEL_Y.to(out.device)
    return ((F.conv2d(out, kx, padding=1) - F.conv2d(target, kx, padding=1)).abs().mean()
            + (F.conv2d(out, ky, padding=1) - F.conv2d(target, ky, padding=1)).abs().mean())


def total_loss(out, target):
    return recon_loss(out, target) + EDGE_WEIGHT * edge_loss(out, target)


# fixed validation material (the same every epoch)
VAL_PAIRS = make_batch("val", 128, np.random.default_rng(SEED_SR + 7))
val_board = [l for l in lines_by_split["val"] if l["source"] == "board"][:VAL_OCR_LINES]
val_iam = [l for l in lines_by_split["val"] if l["source"] == "iam"][:VAL_OCR_LINES // 2]
VAL_OCR = [(l["gt"], degrade_random(l["img"], np.random.default_rng([SEED_SR, i, s]), s), s)
           for i, l in enumerate(val_board + val_iam) for s in SCALES]
assert VAL_OCR, "no validation lines"
print(f"validation: {len(VAL_PAIRS[0])} patches for PSNR, {len(VAL_OCR)} damaged lines for CER "
      f"({len(val_board)} whiteboard + {len(val_iam)} IAM lines x {len(SCALES)} scales)")


@torch.no_grad()
def validate(model):
    model.eval()
    dev = model_device(model)
    outs = []
    for i in range(0, len(VAL_PAIRS[0]), 32):
        outs.append(model(torch.from_numpy(VAL_PAIRS[0][i:i + 32]).to(dev)).cpu().numpy())
    out = np.clip(np.concatenate(outs), 0, 1)
    psnr = float(10 * np.log10(1 / max(np.mean((out - VAL_PAIRS[1]) ** 2), 1e-10)))
    imgs = [sr_line(model, lr, HR_HEIGHT / lr.shape[0]) for _, lr, _ in VAL_OCR]
    val_cer = float(np.mean([cer(gt, t) for (gt, _, _), (t, _) in zip(VAL_OCR, read(imgs))]))
    return psnr, val_cer


def train_sr():
    settings = {k: globals()[k] for k in ["HR_HEIGHT", "MIN_SOURCE_H", "MAX_LABEL_CER", "USE_IAM", "IAM_LINES", "SPLIT_BY",
                                         "MANUAL_SPLIT", "SPLIT_FRACTIONS", "SCALES", "BLUR_SIGMA", "MOTION_BLUR_PROB",
                                         "MOTION_BLUR_LEN", "NOISE_SIGMA", "JPEG_QUALITY", "BRIGHTNESS", "CONTRAST",
                                         "GLARE_PROB", "GLARE_STRENGTH", "CHANNELS", "RES_BLOCKS", "PATCH_WIDTH",
                                         "BATCH_SIZE", "STEPS_PER_EPOCH_SR", "MAX_EPOCHS_SR", "LR_START", "LOSS",
                                         "EDGE_WEIGHT", "IAM_SHARE", "VAL_OCR_LINES", "LR_HALVE_AFTER", "STOP_AFTER", "SEED_SR"]}
    settings = json.loads(json.dumps(settings))                       # tuples -> lists, as stored in the json
    saved = json.loads(CKPT_META.read_text()) if CKPT_META.exists() else None
    if CKPT.exists() and not RETRAIN_SR and saved and saved.get("settings") == settings:
        model = LineSR()
        model.load_state_dict(torch.load(CKPT, map_location="cpu"))
        print(f"loaded {CKPT.name}: epoch {saved['best_epoch']}, validation CER {saved['best_val_cer']:.3f} "
              f"(bicubic {saved['history'][0]['val_cer']:.3f}). Set RETRAIN_SR = True to train again.")
        return model.eval(), pd.DataFrame(saved["history"]), saved
    if CKPT.exists() and not RETRAIN_SR:
        print("the saved model was trained with other settings: training again")

    torch.manual_seed(SEED_SR)
    rng = np.random.default_rng(SEED_SR)
    model = LineSR().to(TRAIN_DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR_START)
    psnr0, cer0 = validate(model)
    history = [dict(epoch=0, train_loss=np.nan, val_psnr=psnr0, val_cer=cer0, lr=LR_START, minutes=0.0)]
    best = dict(cer=cer0, epoch=0, state=copy.deepcopy(model.state_dict()))
    print(f"epoch  0 (= bicubic): val PSNR {psnr0:.2f} dB | val CER {cer0:.3f}")
    bad, t_start = 0, time.time()
    for ep in range(1, MAX_EPOCHS_SR + 1):
        model.train()
        total = 0.0
        for _ in range(STEPS_PER_EPOCH_SR):
            x, y = make_batch("train", BATCH_SIZE, rng)
            out = model(torch.from_numpy(x).to(TRAIN_DEVICE))
            loss = total_loss(out, torch.from_numpy(y).to(TRAIN_DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        v_psnr, v_cer = validate(model)
        lr_now = opt.param_groups[0]["lr"]
        history.append(dict(epoch=ep, train_loss=total / STEPS_PER_EPOCH_SR, val_psnr=v_psnr, val_cer=v_cer,
                            lr=lr_now, minutes=(time.time() - t_start) / 60))
        note = ""
        if v_cer < best["cer"] - 0.002:
            best, bad, note = dict(cer=v_cer, epoch=ep, state=copy.deepcopy(model.state_dict())), 0, "  * best so far"
        else:
            bad += 1
            if bad % LR_HALVE_AFTER == 0:
                for g in opt.param_groups:
                    g["lr"] *= 0.5
                note = f"  (no gain for {bad} epochs: learning rate -> {opt.param_groups[0]['lr']:.1e})"
        print(f"epoch {ep:>2}: train loss {total / STEPS_PER_EPOCH_SR:.4f} | val PSNR {v_psnr:.2f} dB | "
              f"val CER {v_cer:.3f} | lr {lr_now:.1e} | {history[-1]['minutes']:.1f} min{note}")
        if bad >= STOP_AFTER:
            print(f"no better validation CER for {STOP_AFTER} epochs: stopping")
            break
    model.load_state_dict(best["state"])
    torch.save({k: v.cpu() for k, v in best["state"].items()}, CKPT)
    meta = dict(settings=settings, best_epoch=best["epoch"], best_val_cer=best["cer"], history=history,
                weights=n_weights, split=split_table.to_dict("records"))
    CKPT_META.write_text(json.dumps(meta, indent=1, default=float))
    print(f"kept epoch {best['epoch']} (val CER {best['cer']:.3f}; bicubic {cer0:.3f}) -> models/superres/{CKPT.name}")
    if best["epoch"] == 0:
        print("NOTE: no epoch beat bicubic on validation CER, so the saved model IS bicubic")
    return model.eval(), pd.DataFrame(history), meta


sr_model, sr_history, sr_meta = train_sr()
NEW_MODEL["model"] = sr_model.to(INFER_DEVICE).eval()

validation: 128 patches for PSNR, 156 damaged lines for CER (22 whiteboard + 30 IAM lines x 3 scales)
epoch  0 (= bicubic): val PSNR 18.93 dB | val CER 0.299
epoch  1: train loss 0.0817 | val PSNR 19.48 dB | val CER 0.314 | lr 3.0e-04 | 0.8 min
epoch  2: train loss 0.0748 | val PSNR 19.91 dB | val CER 0.361 | lr 3.0e-04 | 1.6 min
epoch  3: train loss 0.0717 | val PSNR 19.35 dB | val CER 0.351 | lr 3.0e-04 | 2.4 min  (no gain for 3 epochs: learning rate -> 1.5e-04)
epoch  4: train loss 0.0686 | val PSNR 19.51 dB | val CER 0.344 | lr 1.5e-04 | 3.2 min
epoch  5: train loss 0.0672 | val PSNR 18.65 dB | val CER 0.347 | lr 1.5e-04 | 4.0 min
epoch  6: train loss 0.0683 | val PSNR 19.93 dB | val CER 0.339 | lr 1.5e-04 | 4.8 min  (no gain for 6 epochs: learning rate -> 7.5e-05)
epoch  7: train loss 0.0669 | val PSNR 19.76 dB | val CER 0.342 | lr 7.5e-05 | 5.6 min
epoch  8: train loss 0.0670 | val PSNR 19.85 dB | val CER 0.333 | lr 7.5e-05 | 6.4 min
no better validation CER for 8 epochs: stoppin

In [24]:
h = sr_history
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(h.epoch, h.train_loss, marker="o", ms=3); ax[0].set_title("training loss (lower = closer to sharp)")
ax[1].plot(h.epoch, h.val_psnr, marker="o", ms=3); ax[1].set_title("validation PSNR, dB (secondary)")
ax[2].plot(h.epoch, h.val_cer, marker="o", ms=3, label="model")
ax[2].axhline(h.val_cer.iloc[0], color="grey", ls="--", label="bicubic (epoch 0)")
best_ep = int(sr_meta["best_epoch"])
ax[2].scatter([best_ep], [h.set_index("epoch").val_cer[best_ep]], s=120, facecolors="none", edgecolors="red", lw=2,
              label="kept")
ax[2].set_title("validation CER from PaddleOCR (main; lower = better)"); ax[2].legend()
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "handwriting_sr_training.png", dpi=150); plt.show()

<Figure size 1500x360 with 3 Axes>

## E6. Evaluation on held-out test lines

Every test line is damaged once per scale with a **fixed** random seed, so every method gets exactly the same small images. Then:
- `none` gives the small line to PaddleOCR as it is (PaddleOCR resizes it itself);
- every other method enlarges it to `HR_HEIGHT` first;
- `hr` reads the original sharp line: the best possible score (upper bound).

Main scores (PaddleOCR): **CER**, **WER** (word error rate, capped at 1), **exact match** (whole line right, case and spacing ignored), **confidence** (PaddleOCR's own score). Also **time**: ms per line for the enlargement (CPU) and for reading. Secondary: PSNR and SSIM against the sharp line (higher = closer pixels).

In [25]:
test_sets = {src: [l for l in lines_by_split["test"] if l["source"] == src][:MAX_TEST_LINES] for src in ("board", "iam")}
cases = []
for src, ls in test_sets.items():
    for i, l in enumerate(ls):
        for s in SCALES:
            lr = degrade_random(l["img"], np.random.default_rng([SEED_SR + 1000, i, s, src == "iam"]), s)
            cases.append(dict(source=src, photo=l["photo"], board=l["board"], gt=l["gt"], scale=s, lr=lr, hr=l["img"]))
print(f"test: {len(test_sets['board'])} whiteboard lines + {len(test_sets['iam'])} IAM lines, x {len(SCALES)} scales "
      f"= {len(cases)} damaged lines per method")

rows, timing = [], []
for m in EVAL_METHODS + ["hr"]:
    t0 = time.time()
    if m == "hr":
        outs, sr_ms = [c["hr"] for c in cases], [0.0] * len(cases)
    else:
        if m != "none":
            enlarge_any(cases[0]["lr"], HR_HEIGHT / cases[0]["lr"].shape[0], m)      # warm-up (model loading)
        outs, sr_ms = [], []
        for c in cases:
            t = time.perf_counter()
            outs.append(enlarge_any(c["lr"], HR_HEIGHT / c["lr"].shape[0], m))
            sr_ms.append((time.perf_counter() - t) * 1000)
    t = time.perf_counter()
    reads = read(outs)
    ocr_ms = (time.perf_counter() - t) * 1000 / len(outs)
    for c, out, ms, (txt, conf) in zip(cases, outs, sr_ms, reads):
        r = dict(source=c["source"], photo=c["photo"], gt=c["gt"], scale=c["scale"], method=m, read=txt,
                 confidence=conf, sr_ms=ms, ocr_ms=ocr_ms, **text_scores(c["gt"], txt))
        if m not in ("none", "hr") and out.shape[0] == c["hr"].shape[0]:
            a, b = same_size_y(out, c["hr"])
            r.update(psnr=psnr_y(a, b), ssim=ssim_y(a, b))
        rows.append(r)
    print(f"{m:>16}: {time.time() - t0:5.0f} s")
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "handwriting_sr_test_results.csv", index=False)

order = [m for m in EVAL_METHODS + ["hr"]]
for src in ("board", "iam"):
    part = res[res.source == src]
    if part.empty:
        continue
    print(f"\n=== {'WHITEBOARD' if src == 'board' else 'IAM'} test lines: CER by scale (lower = better) ===")
    display(part.pivot_table(index="method", columns="scale", values="cer", aggfunc="mean").reindex(order).round(3))
    summary_tab = part.groupby("method").agg(CER=("cer", "mean"), WER=("wer", "mean"), exact_match=("exact", "mean"),
                                             confidence=("confidence", "mean"), sr_ms_per_line=("sr_ms", "mean"),
                                             ocr_ms_per_line=("ocr_ms", "mean"), PSNR=("psnr", "mean"),
                                             SSIM=("ssim", "mean")).reindex(order)
    print("all scales together:")
    display(summary_tab.round(3))

test: 24 whiteboard lines + 100 IAM lines, x 3 scales = 372 damaged lines per method
            none:    38 s
         bicubic:    35 s
 bicubic+sharpen:    35 s
           espcn:    35 s
          fsrcnn:    35 s
      realesrgan:   489 s
     espcn-board:    36 s
  handwriting-sr:    83 s
              hr:    34 s

=== WHITEBOARD test lines: CER by scale (lower = better) ===


scale,2,3,4
method,,,
none,0.212,0.462,0.733
bicubic,0.227,0.490,0.768
bicubic+sharpen,0.230,0.531,0.814
espcn,0.221,0.479,0.774
fsrcnn,0.228,0.476,0.735
realesrgan,0.286,0.593,0.864
espcn-board,0.304,0.641,0.844
handwriting-sr,0.229,0.486,0.772
hr,0.028,0.028,0.028


all scales together:


,CER,WER,exact_match,confidence,sr_ms_per_line,ocr_ms_per_line,PSNR,SSIM
method,,,,,,,,
none,0.469,0.712,0.097,0.672,0.000,100.919,NaN,NaN
bicubic,0.495,0.736,0.097,0.650,0.040,92.997,20.621,0.734
bicubic+sharpen,0.525,0.757,0.083,0.627,0.184,91.116,20.642,0.717
espcn,0.491,0.726,0.083,0.668,0.579,91.095,20.625,0.740
fsrcnn,0.480,0.719,0.083,0.659,1.131,90.601,20.626,0.738
realesrgan,0.581,0.788,0.042,0.714,301.727,91.704,20.075,0.784
espcn-board,0.596,0.799,0.042,0.566,1.107,92.354,20.691,0.776
handwriting-sr,0.496,0.736,0.111,0.657,33.806,90.737,20.621,0.734
hr,0.028,0.146,0.542,0.960,0.000,92.066,NaN,NaN



=== IAM test lines: CER by scale (lower = better) ===


scale,2,3,4
method,,,
none,0.160,0.189,0.308
bicubic,0.165,0.202,0.329
bicubic+sharpen,0.169,0.207,0.347
espcn,0.160,0.196,0.312
fsrcnn,0.166,0.204,0.343
realesrgan,0.155,0.238,0.382
espcn-board,0.152,0.216,0.367
handwriting-sr,0.165,0.204,0.329
hr,0.136,0.136,0.136


all scales together:


,CER,WER,exact_match,confidence,sr_ms_per_line,ocr_ms_per_line,PSNR,SSIM
method,,,,,,,,
none,0.219,0.538,0.033,0.850,0.000,100.919,NaN,NaN
bicubic,0.232,0.565,0.030,0.847,0.181,92.997,18.103,0.697
bicubic+sharpen,0.241,0.584,0.023,0.842,0.399,91.116,18.327,0.685
espcn,0.223,0.538,0.030,0.845,1.695,91.095,18.079,0.708
fsrcnn,0.238,0.550,0.037,0.838,3.182,90.601,18.102,0.698
realesrgan,0.258,0.561,0.037,0.846,1432.831,91.704,16.822,0.752
espcn-board,0.245,0.554,0.037,0.835,4.326,92.354,18.188,0.752
handwriting-sr,0.233,0.564,0.033,0.848,153.088,90.737,18.103,0.697
hr,0.136,0.419,0.070,0.929,0.000,92.066,NaN,NaN


In [26]:
def boot_gain(df, method, baseline, n_boot=4000, seed=0):
    """Mean CER gain of `method` over `baseline` (positive = method better), with a 95% CI that
    resamples whole photos (lines from one photo are not independent)."""
    w = df[df.method.isin([method, baseline])].pivot_table(index=["photo", "gt", "scale"], columns="method",
                                                          values="cer", aggfunc="mean").dropna().reset_index()
    if w.empty or method == baseline:
        return dict(n=0, gain=np.nan, low=np.nan, high=np.nan, better=0, worse=0)
    w["d"] = w[baseline] - w[method]
    groups = [g.d.values for _, g in w.groupby("photo")]
    rng = np.random.default_rng(seed)
    boots = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return dict(n=len(w), gain=w.d.mean(), low=lo, high=hi, better=int((w.d > 1e-3).sum()), worse=int((w.d < -1e-3).sum()))


small_scales = [s for s in SCALES if s >= 3]
rows = []
for src in ("board", "iam"):
    part = res[res.source == src]
    for m in [x for x in EVAL_METHODS if x != "none"]:
        for base in [b for b in ("none", "bicubic") if b != m]:
            for label, sc in [("x3+x4", small_scales), ("x2", [2]), ("all", list(SCALES))]:
                r = boot_gain(part[part.scale.isin(sc)], m, base)
                rows.append(dict(source=src, method=m, vs=base, scales=label, **r))
gains = pd.DataFrame(rows)
gains.to_csv(OUT_DIR / "handwriting_sr_gains.csv", index=False)
print("CER gain over the baseline (positive = better), 95% CI by resampling photos:")
display(gains[gains.scales == "x3+x4"].round(3))

g = gains.set_index(["source", "method", "vs", "scales"])
print("\nVERDICT (whiteboard test lines, rule set before training):")
if ("board", "handwriting-sr", "none", "x3+x4") in g.index and g.loc[("board", "handwriting-sr", "none", "x3+x4"), "n"] > 0:
    vn, vb = g.loc[("board", "handwriting-sr", "none", "x3+x4")], g.loc[("board", "handwriting-sr", "bicubic", "x3+x4")]
    v2 = g.loc[("board", "handwriting-sr", "none", "x2")]
    ok = (vn.gain >= GAIN_NEEDED_SR and vn.low > 0 and vb.gain >= GAIN_NEEDED_SR and vb.low > 0 and v2.gain >= -0.01)
    print(f"  x3+x4 vs none:    {vn.gain:+.3f} [{vn.low:+.3f}, {vn.high:+.3f}] (need >= +{GAIN_NEEDED_SR}, CI above 0)")
    print(f"  x3+x4 vs bicubic: {vb.gain:+.3f} [{vb.low:+.3f}, {vb.high:+.3f}] (need >= +{GAIN_NEEDED_SR}, CI above 0)")
    print(f"  x2 vs none:       {v2.gain:+.3f} (must be >= -0.01)")
    print("  ->", "handwriting-sr HELPS: consider it for the pipeline" if ok else "handwriting-sr does NOT meet the rule")
else:
    print("  no whiteboard test lines: cannot judge (change SPLIT_BY / MANUAL_SPLIT)")

CER gain over the baseline (positive = better), 95% CI by resampling photos:


,source,method,vs,scales,n,gain,low,high,better,worse
0,board,bicubic,none,x3+x4,48,-0.031,-0.051,-0.011,8,20
3,board,bicubic+sharpen,none,x3+x4,48,-0.075,-0.109,-0.048,3,27
6,board,bicubic+sharpen,bicubic,x3+x4,48,-0.044,-0.063,-0.030,4,22
9,board,espcn,none,x3+x4,48,-0.029,-0.050,-0.012,7,17
12,board,espcn,bicubic,x3+x4,48,0.002,-0.011,0.016,14,11
15,board,fsrcnn,none,x3+x4,48,-0.008,-0.015,-0.000,11,12
18,board,fsrcnn,bicubic,x3+x4,48,0.023,0.004,0.043,20,9
21,board,realesrgan,none,x3+x4,48,-0.131,-0.178,-0.067,9,31
24,board,realesrgan,bicubic,x3+x4,48,-0.100,-0.152,-0.034,14,28
27,board,espcn-board,none,x3+x4,48,-0.145,-0.191,-0.103,4,28



VERDICT (whiteboard test lines, rule set before training):
  x3+x4 vs none:    -0.031 [-0.055, -0.008] (need >= +0.02, CI above 0)
  x3+x4 vs bicubic: -0.000 [-0.005, +0.005] (need >= +0.02, CI above 0)
  x2 vs none:       -0.017 (must be >= -0.01)
  -> handwriting-sr does NOT meet the rule


In [27]:
# what the methods produce and what PaddleOCR reads (left part of the line shown so letters stay visible)
SHOW = [m for m in ["bicubic", "handwriting-sr"] if m in EVAL_METHODS]
assert "none" in EVAL_METHODS, "keep 'none' in EVAL_METHODS (it is the baseline)"
examples = [l for l in test_sets["board"]][:2] or test_sets["iam"][:2]
lookup = res.set_index(["photo", "gt", "scale", "method"]).sort_index()
for ex in examples:
    fig, axes = plt.subplots(2 + len(SHOW), len(SCALES), figsize=(5.2 * len(SCALES), 1.25 * (2 + len(SHOW))))
    view = slice(0, min(ex["img"].shape[1], 6 * HR_HEIGHT))
    for j, s in enumerate(SCALES):
        c = next(c for c in cases if c["photo"] == ex["photo"] and c["gt"] == ex["gt"] and c["scale"] == s)
        f = HR_HEIGHT / c["lr"].shape[0]
        panels = [("HR (sharp target)", ex["img"], "hr"),
                  (f"LR x{s} ({c['lr'].shape[0]} px), big pixels",
                   cv2.resize(c["lr"], (int(round(c["lr"].shape[1] * f)), HR_HEIGHT), interpolation=cv2.INTER_NEAREST), "none")]
        panels += [(m, enlarge_any(c["lr"], f, m), m) for m in SHOW]
        for i, (title, im, m) in enumerate(panels):
            r = lookup.loc[(ex["photo"], ex["gt"], s, m)]
            r = r.iloc[0] if isinstance(r, pd.DataFrame) else r
            axes[i, j].imshow(rgb(im[:, view]))
            axes[i, j].set_title(f"{title}\nread: \"{r['read'][:40]}\"  CER {r['cer']:.2f}", fontsize=8)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    fig.suptitle(f"truth: \"{ex['gt']}\" ({ex['photo']})", fontsize=10)
    plt.tight_layout(); plt.show()

<Figure size 1560x500 with 12 Axes>

<Figure size 1560x500 with 12 Axes>

### E7. Real far-away photos (no synthetic damage)

The real test: every *Scale and Resolution Variation* photo whose board is in the **test** split. Lines are detected at full resolution, lines shorter than `HR_HEIGHT` are enlarged (max x4) by each method, and each ground-truth line is matched with its best reading. Grouped by the detector's box height, like Experiment 1.

In [28]:
if not RUN_REAL_FAR_TEST:
    print("skipped (RUN_REAL_FAR_TEST = False)")
else:
    test_boards = {l["board"] for l in lines_by_split["test"] if l["source"] == "board"}
    test_photos = {l["photo"] for l in lines_by_split["test"] if l["source"] == "board"}
    far = images[(images.condition == "Scale and Resolution Variation")
                 & images.image_id.map(lambda n: (n.split("_")[0] in test_boards) if SPLIT_BY == "board" else (n in test_photos))]
    far = far.image_id.tolist()
    print(f"{len(far)} real far-away test photos (boards: {', '.join(sorted(test_boards))})")
    rows = []
    for k, name in enumerate(far, 1):
        img, gts = load(name), gt_lines(name)
        crops = [quad_crop(img, p) for p in detect_full(img)]
        reads = {}
        for m in REAL_TEST_METHODS:
            ins = [enlarge_any(c, float(np.clip(HR_HEIGHT / c.shape[0], 1, 4)), m) for c in crops]
            reads[m] = [t for t, _ in read(ins)]
        for g in gts:
            band_h = crops[min(range(len(crops)), key=lambda i: cer(g, reads["none"][i]))].shape[0] if crops else 0
            for m in REAL_TEST_METHODS:
                txt = min(reads[m], key=lambda t: cer(g, t), default="")
                rows.append(dict(photo=name, gt=g, method=m, box_h=band_h, read=txt, scale=0, **text_scores(g, txt)))
        print(f"[{k}/{len(far)}] {name}: {len(crops)} boxes, {len(gts)} lines")
    real2 = pd.DataFrame(rows)
    if real2.empty:
        print("no far-away photos on the test boards")
    else:
        real2["band"] = pd.cut(real2.box_h, [0, 16, 20, 24, 32, 10_000], right=False,
                               labels=["0-16", "16-20", "20-24", "24-32", ">=32"])
        real2.to_csv(OUT_DIR / "handwriting_sr_real_far.csv", index=False)
        display(real2.pivot_table(index="method", columns="band", values="cer", aggfunc="mean", observed=False)
                .reindex(index=REAL_TEST_METHODS, columns=real2.band.cat.categories).round(3))
        print("lines per band:", real2[real2.method == "none"].groupby("band", observed=False).size().to_dict())
        for m in [x for x in REAL_TEST_METHODS if x != "none"]:
            for label, part in [("16-24 px", real2[(real2.box_h >= 16) & (real2.box_h < 24)]), ("all lines", real2)]:
                r = boot_gain(part, m, "none")
                if r["n"] == 0:
                    print(f"{m:>16} vs none, {label:>9}: no lines")
                    continue
                print(f"{m:>16} vs none, {label:>9}: {r['gain']:+.3f} [{r['low']:+.3f}, {r['high']:+.3f}] "
                      f"({r['n']} lines, {r['better']} better / {r['worse']} worse)")

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


5 real far-away test photos (boards: quiz)


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[1/5] quiz_08.png: 17 boxes, 6 lines


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[2/5] quiz_09.png: 17 boxes, 6 lines


... (2 outputs left out)

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[4/5] quiz_11.png: 19 boxes, 6 lines
[5/5] quiz_26.png: 19 boxes, 6 lines


,0-16,16-20,20-24,24-32,>=32
method,,,,,
none,NaN,0.773,NaN,NaN,0.142
bicubic,NaN,0.773,NaN,NaN,0.142
bicubic+sharpen,NaN,0.727,NaN,NaN,0.142
espcn,NaN,0.773,NaN,NaN,0.149
espcn-board,NaN,0.727,NaN,NaN,0.153
handwriting-sr,NaN,0.773,NaN,NaN,0.142


lines per band: {'0-16': 0, '16-20': 1, '20-24': 0, '24-32': 0, '>=32': 29}
         bicubic vs none,  16-24 px: +0.000 [+0.000, +0.000] (1 lines, 0 better / 0 worse)
         bicubic vs none, all lines: +0.000 [+0.000, +0.000] (30 lines, 0 better / 0 worse)
 bicubic+sharpen vs none,  16-24 px: +0.045 [+0.045, +0.045] (1 lines, 1 better / 0 worse)
 bicubic+sharpen vs none, all lines: +0.002 [+0.000, +0.005] (30 lines, 1 better / 0 worse)
           espcn vs none,  16-24 px: +0.000 [+0.000, +0.000] (1 lines, 0 better / 0 worse)
           espcn vs none, all lines: -0.006 [-0.018, +0.000] (30 lines, 0 better / 2 worse)
     espcn-board vs none,  16-24 px: +0.045 [+0.045, +0.045] (1 lines, 1 better / 0 worse)
     espcn-board vs none, all lines: -0.009 [-0.026, +0.000] (30 lines, 1 better / 2 worse)
  handwriting-sr vs none,  16-24 px: +0.000 [+0.000, +0.000] (1 lines, 0 better / 0 worse)
  handwriting-sr vs none, all lines: +0.000 [+0.000, +0.000] (30 lines, 0 better / 0 worse)


## E8. How to read the results

- **Training (E5)**: did validation CER go *below* the dashed bicubic line? If the kept epoch is 0, training never beat bicubic at reading and the saved model is just bicubic.
- **Main result (E6)**: the whiteboard table and the verdict. `hr` is the ceiling: the gap between `none` and `hr` is what any enlargement could possibly win back.
- **PSNR/SSIM vs CER**: if PSNR/SSIM go up but CER does not go down, the model makes lines *look* closer to the original without making them easier to *read* (as in Experiment 1).
- **Real photos (E7)**: synthetic damage is only an imitation of a far-away photo; this is the test that matters for the app.

**Known limits**
- Only 5 boards (= 5 writers/sessions) and few different texts outside the `lecture` board: the whiteboard test set is small; read the confidence intervals, not only the means.
- IAM is pen on paper; whiteboard marker strokes are thicker and the background differs.
- The recogniser itself is not changed here. Fine-tuning PaddleOCR on small, damaged lines is the competing approach and should be judged with the same test.

---
# Experiment 1 (kept for reference): ESPCN x4 fine-tuned on IAM / our photos

The cells below are the previous experiment **with the outputs of its last run**. The next cell stops "Run All" so it is not re-run by accident; set `RUN_EXPERIMENT_1 = True` there to re-run it (about 30 min).

In [29]:
RUN_EXPERIMENT_1 = False   # True: also re-run Experiment 1 below (about 30 min)
if not RUN_EXPERIMENT_1:
    raise SystemExit("Stopped on purpose: Experiment 1 below is kept for reference (set RUN_EXPERIMENT_1 = True to re-run it).")

SystemExit: Stopped on purpose: Experiment 1 below is kept for reference (set RUN_EXPERIMENT_1 = True to re-run it).

<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Train our own super-resolution CNN (ESPCN x4) and compare it fairly

The pretrained ESPCN (trained on ordinary photos) did not help our recogniser (`super_resolution_pretrained.ipynb`). Here we **train the same small CNN on handwriting** and test whether that changes the answer.

Two trained versions, both starting from the pretrained ESPCN x4 weights (fine-tuning):

| Model | Trained on | Question it answers |
|---|---|---|
| `espcn-iam` | IAM handwriting lines (pen on paper), made whiteboard-like with our augmentation policy | does handwriting in general help? |
| `espcn-board` | **our own whiteboard photos** (`lecture_*`, `animal_*`) | does *our* kind of image (marker, board, phone camera) help more? |

**Fair test, decided before training**
- Training never sees the test boards: train on `lecture_*` + `animal_*`, test on `television_*`, `cnn_*`, `quiz_*` (different boards, different handwriting sessions). IAM is never in the test.
- Every method is scored on the same lines: `none` (give the small line to the recogniser as is), `bicubic+sharpen` (current pipeline), `espcn` (pretrained), `espcn-iam`, `espcn-board`.
- **Success rule**: a trained model "works" only if, on the **real far-away photos**, it beats `none` by at least **0.02 CER** for line boxes of 16-24 px, and is not worse than `none` by more than 0.01 for bigger lines.

How training works (same idea as the week 10 inpainting / autoencoder slides): take a sharp line, damage it ourselves with `degrade` (shrink to 16-24 px + blur + JPEG, exactly as in the test), and train the CNN to give back the sharp version (L1 loss, backprop). No labels needed.

Kernel: **Python 3.12 (whiteboard-ocr)**. Runs on a Mac (Apple GPU "mps" if available, else CPU). Rough timing on an M-series Mac: ~10 min to find the training lines (cached after the first run), ~5-10 min per model, ~20 min for the evaluation. Trained weights are saved to `models/superres/`, so a re-run skips training (`RETRAIN`).

## 0b. Setup and settings

In [ ]:
import os
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import copy
import json
import sys
import time
from pathlib import Path

import cv2
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from paddleocr import TextDetection, TextRecognition


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "notebooks").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the whiteboard-ocr repo folder")


REPO = find_repo_root()
DATA = REPO / "data" / "board2notes_v1"
IMG_DIR = DATA / "images"
LABELS = DATA / "board2notes_labels_v1_textonly.xlsx"
IAM_DIR = REPO / "data" / "iam_line_paddle"                 # IAM lines (train/, val/, train.txt, val.txt)
SR_DIR = REPO / "models" / "superres"
OUT_DIR = REPO / "outputs" / "super_resolution"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO / "augmentation"))
from ocr_augmentation import augment, load_policy               # noqa: E402  (our IAM -> whiteboard augmentation)

# ---- which photos train, which test (split by BOARD, so test handwriting is never seen in training) ----
TRAIN_PREFIXES = ("lecture", "animal")
TEST_PREFIXES = ("television", "cnn", "quiz")

# ---- training ----
SCALE = 4                    # x4: a 16-24 px line becomes 64-96 px
HR_PATCH = 64                # sharp target patch (px); the input patch is HR_PATCH / SCALE = 16 px
MARGIN = 32                  # extra context left/right when damaging a patch (so blur/JPEG at the edges is realistic)
LR_BOX_RANGE = (16, 24)      # line-box heights the training inputs are shrunk to: the zone that matters
MAX_TARGET_FRACTION = 0.9    # a sharp target is at most 90% of the source line's height (always shrunk a bit = cleaner than raw pixels)
MIN_TRAIN_CROP_H = 72        # our photos: lines at least this tall (72 x 0.9 -> 64 px targets)
IAM_LINE_H = 112             # IAM lines are stored at this height after padding (see part 2)
BATCH = 64
MAX_EPOCHS = 60             # upper limit; training normally stops earlier (early stopping, below)
STEPS_PER_EPOCH = 300        # 1 "epoch" = 300 batches of fresh random pairs (the data is generated, so there is no fixed pass)
LEARNING_RATE = 1e-4         # fine-tuning from the pretrained weights
MIN_DELTA = 0.01             # dB: a smaller rise in validation PSNR does not count as an improvement
LR_PATIENCE = 3              # no improvement for 3 epochs -> halve the learning rate (smaller, finer steps)
STOP_PATIENCE = 8            # no improvement for 8 epochs -> stop and keep the best epoch
IAM_TRAIN_LINES = 1500       # IAM lines kept in memory for training (more = more variety, more RAM)
IAM_VAL_LINES = 200
VAL_PATCHES = 512
RETRAIN = False              # False: if models/superres/<name>_x4.pt exists, load it instead of training again
SEED = 0
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# ---- evaluation (same definitions as super_resolution_pretrained.ipynb) ----
TARGET_H, MAX_SCALE = 64, 4.0
DET_ARGS = dict(thresh=0.20, box_thresh=0.40, unclip_ratio=2.00)
HEIGHTS = [16, 20, 24, 28, 32]        # simulated test: line-box heights
MAX_LINES = 80
MIN_REF_H, MAX_REF_CER = 36, 0.15
BANDS = [(0, 16), (16, 20), (20, 24), (24, 32), (32, 10_000)]   # real photos: line-box height bands (px)
GAIN_NEEDED = 0.02                    # success rule (see top)

torch.manual_seed(SEED)
labels = pd.read_excel(LABELS, sheet_name="Line Labels", keep_default_na=False)
images = pd.read_excel(LABELS, sheet_name="Images", keep_default_na=False)
train_photos = [n for n in images.image_id if n.startswith(TRAIN_PREFIXES)]
test_photos = [n for n in images.image_id if n.startswith(TEST_PREFIXES)]
assert not set(train_photos) & set(test_photos)


def gt_lines(name):
    return labels.loc[labels.image_id == name, "ground_truth"].tolist()


det_model = TextDetection(model_name="PP-OCRv5_mobile_det")
rec_model = TextRecognition(model_name="PP-OCRv6_medium_rec")
print(f"device: {DEVICE} | {len(train_photos)} training photos ({', '.join(TRAIN_PREFIXES)}), "
      f"{len(test_photos)} test photos ({', '.join(TEST_PREFIXES)})")

<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv5_mobile_det`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


device: mps | 16 training photos (lecture, animal), 92 test photos (television, cnn, quiz)


### Helpers (identical to `super_resolution_pretrained.ipynb`, so the numbers are comparable)

In [ ]:
def load(name):
    img = cv2.imread(str(IMG_DIR / name))
    assert img is not None, f"cannot read {IMG_DIR / name}"
    return img


def degrade(crop, h, realistic=True):
    """Make a clear line look like it was photographed from far away: `h` px tall (blur + resize + JPEG)."""
    s = h / crop.shape[0]
    w = max(1, int(round(crop.shape[1] * s)))
    img = crop
    if realistic:
        img = cv2.GaussianBlur(img, (0, 0), sigmaX=max(0.3, 0.5 / s))
    small = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
    if realistic:
        ok, buf = cv2.imencode(".jpg", small, [cv2.IMWRITE_JPEG_QUALITY, 70])
        small = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return small


_sr_cache = {}


def _sr(name, k):
    key = (name, k)
    if key not in _sr_cache:
        m = cv2.dnn_superres.DnnSuperResImpl_create()
        m.readModel(str(SR_DIR / f"{name.upper()}_x{k}.pb"))
        m.setModel(name.lower(), k)
        _sr_cache[key] = m
    return _sr_cache[key]


TRAINED = {}          # name -> trained ESPCN (filled in part 3), used by enlarge()


def enlarge(img, s, method):
    """Enlarge by factor s: 'none', 'bicubic', 'bicubic+sharpen', pretrained 'espcn', or a trained model name."""
    if s <= 1.05 or method == "none":
        return img
    size = (int(round(img.shape[1] * s)), int(round(img.shape[0] * s)))
    if method == "bicubic":
        return cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
    if method == "bicubic+sharpen":
        up = cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
        blur = cv2.GaussianBlur(up, (0, 0), sigmaX=0.8 * s)
        return cv2.addWeighted(up, 1.6, blur, -0.6, 0)
    if method in TRAINED:          # our trained models are x4 only: enlarge x4, then shrink to the wanted size
        up, k = sr_trained(img, TRAINED[method]), SCALE
    else:                          # pretrained: smallest of x2/x3/x4 that is big enough (as in the pretrained notebook)
        k = int(min(4, max(2, np.ceil(s))))
        up = _sr(method, k).upsample(img)
    return cv2.resize(up, size, interpolation=cv2.INTER_AREA if k > s else cv2.INTER_CUBIC)


def read(imgs):
    if not imgs:
        return []
    return [(r["rec_text"], float(r["rec_score"])) for r in rec_model.predict(list(imgs))]


def cer(gt, text):
    return min(jiwer.cer(gt.lower(), text.lower() or " "), 1.0)


def line_height(poly):
    p = np.asarray(poly, float).reshape(-1, 2)
    if len(p) != 4:
        return float(np.ptp(p[:, 1]))
    return float((np.linalg.norm(p[0] - p[3]) + np.linalg.norm(p[1] - p[2])) / 2)


def quad_crop(img, poly):
    p = np.float32(np.asarray(poly, float).reshape(-1, 2))
    if len(p) != 4:
        p = np.float32(cv2.boxPoints(cv2.minAreaRect(p)))
    s_, d_ = p.sum(1), np.diff(p, axis=1).ravel()
    p = np.float32([p[s_.argmin()], p[d_.argmin()], p[s_.argmax()], p[d_.argmax()]])
    w = int(round(max(np.linalg.norm(p[0] - p[1]), np.linalg.norm(p[3] - p[2]))))
    h = int(round(max(np.linalg.norm(p[0] - p[3]), np.linalg.norm(p[1] - p[2]))))
    M = cv2.getPerspectiveTransform(p, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))
    return cv2.warpPerspective(img, M, (max(w, 1), max(h, 1)), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def detect_full(img):
    res = det_model.predict(img, limit_side_len=max(img.shape[:2]), limit_type="max", **DET_ARGS)
    return [np.asarray(p, float).reshape(-1, 2) for p in res[0]["dt_polys"]]


def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 1. ESPCN in PyTorch, starting from the pretrained weights

The same network as Part B of the pretrained notebook: conv 5x5 (64) -> ReLU -> conv 3x3 (32) -> ReLU -> conv 3x3 (16 = 4x4) -> pixel shuffle -> tanh.
We copy the pretrained weights out of `ESPCN_x4.pb` into it, then **check** it gives the same output as OpenCV's ESPCN before training anything.

In [ ]:
def _varint(b, i):
    r = s = 0
    while True:
        x = b[i]; i += 1
        r |= (x & 0x7F) << s; s += 7
        if x < 0x80:
            return r, i


def _fields(b):
    i = 0
    while i < len(b):
        key, i = _varint(b, i); f, w = key >> 3, key & 7
        if w == 0:
            v, i = _varint(b, i)
        elif w == 1:
            v = b[i:i + 8]; i += 8
        elif w == 2:
            n, i = _varint(b, i); v = b[i:i + n]; i += n
        elif w == 5:
            v = b[i:i + 4]; i += 4
        else:
            raise ValueError(f"unsupported wire type {w}")
        yield f, w, v


def read_tf_weights(pb_path):
    """{name: array} for every constant in a frozen TensorFlow graph (.pb), without TensorFlow."""
    out = {}
    for f, _, node in _fields(Path(pb_path).read_bytes()):
        if f != 1:
            continue
        name = op = tensor = None
        for nf, _, nv in _fields(node):
            if nf == 1:
                name = nv.decode()
            elif nf == 2:
                op = nv.decode()
            elif nf == 5:
                key = val = None
                for ef, _, ev in _fields(nv):
                    if ef == 1:
                        key = ev.decode()
                    elif ef == 2:
                        val = ev
                if key == "value" and val is not None:
                    tensor = next((av for af, _, av in _fields(val) if af == 8), None)
        if op != "Const" or tensor is None:
            continue
        shape, content, floats = [], b"", []
        for tf_, tw, tv in _fields(tensor):
            if tf_ == 2:
                for sf, _, sv in _fields(tv):
                    if sf == 2:
                        shape += [dv for df, _, dv in _fields(sv) if df == 1]
            elif tf_ == 4:
                content = tv
            elif tf_ == 5:
                floats += list(np.frombuffer(tv, "<f4")) if tw == 2 else [np.frombuffer(tv, "<f4")[0]]
        arr = np.frombuffer(content, "<f4") if content else np.array(floats, np.float32)
        if shape and arr.size == 1 and np.prod(shape) > 1:
            arr = np.full(int(np.prod(shape)), arr[0], np.float32)
        out[name] = arr.reshape(shape) if shape else arr
    return out


class ESPCN(nn.Module):
    def __init__(self, r=SCALE):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 5, padding=2)
        self.conv2 = nn.Conv2d(64, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, r * r, 3, padding=1)
        self.shuffle = nn.PixelShuffle(r)

    def forward(self, x):                      # x: (batch, 1, h, w) brightness in [0, 1]
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        return torch.tanh(self.shuffle(self.conv3(x)))


def load_pretrained(model):
    # TensorFlow stores conv weights as (kh, kw, in, out); PyTorch wants (out, in, kh, kw)
    W = read_tf_weights(SR_DIR / f"ESPCN_x{SCALE}.pb")
    with torch.no_grad():
        for conv, f, b in [(model.conv1, "f1", "b1"), (model.conv2, "f2", "b2"), (model.conv3, "f3", "b3")]:
            conv.weight.copy_(torch.from_numpy(W[f].transpose(3, 2, 0, 1).copy()))
            conv.bias.copy_(torch.from_numpy(W[b].copy()))
    return model


def sr_trained(img_bgr, model):
    # like OpenCV's ESPCN: the CNN enlarges brightness (Y); colour (Cr, Cb) is enlarged with bicubic
    ycc = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    y = torch.from_numpy(np.ascontiguousarray(ycc[:, :, 0], dtype=np.float32)[None, None] / 255)
    with torch.no_grad():
        out = model(y).cpu().numpy()[0, 0]
    big = cv2.resize(ycc, (img_bgr.shape[1] * SCALE, img_bgr.shape[0] * SCALE), interpolation=cv2.INTER_CUBIC)
    big[:, :, 0] = np.clip(out * 255 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(big, cv2.COLOR_YCrCb2BGR)


# check: our PyTorch ESPCN with the copied weights == OpenCV's ESPCN
check_model = load_pretrained(ESPCN()).eval()
demo = degrade(load("lecture_02.png")[1290:1370, 1985:2480], 20)
y_ours = cv2.cvtColor(sr_trained(demo, check_model), cv2.COLOR_BGR2YCrCb)[:, :, 0].astype(float)
y_cv = cv2.cvtColor(_sr("espcn", SCALE).upsample(demo), cv2.COLOR_BGR2YCrCb)[:, :, 0].astype(float)
diff = np.abs(y_ours - y_cv).mean()
print(f"PyTorch vs OpenCV ESPCN x{SCALE}: mean |difference| = {diff:.2f} grey levels "
      f"({sum(p.numel() for p in check_model.parameters())} weights)")
assert diff < 3, "weights were not copied correctly: stop here"

PyTorch vs OpenCV ESPCN x4: mean |difference| = 0.00 grey levels (24752 weights)


## 2. Training data: (damaged input, sharp target) pairs

**Sources**
- **Our photos** (`lecture_*`, `animal_*` only): every detected line at least `MIN_TRAIN_CROP_H` px tall, cut straight with `quad_crop`. Real marker, real board, real camera. Lines from a few whole photos are held out to pick the best epoch.
- **IAM**: scanned pen lines. The ink takes a different share of an IAM line than of our detector's boxes, so we first **pad or crop each IAM line** until the ink takes the same share as in our boxes. Otherwise "a 20 px line" would mean different-sized letters in training and in the test. Our augmentation policy (**geometry + lighting only**) makes them look a bit more like a photographed board.

**One training pair**: pick a line-box height h = 16..24 px, take a window of the line scaled to H = 4h, cut a 64x64 sharp patch containing ink (= target), and make the input by running the *same* `degrade` as the test on it down to H/4, giving 16x16. Blur and JPEG come only from `degrade`, so training damage = test damage.

In [ ]:
def ink_rows(gray):
    # first and last row that contain ink (same measure as in the pretrained notebook)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    rows = bw.mean(1) / 255
    on = np.nonzero(rows >= 0.25 * rows.max())[0]
    return (int(on[0]), int(on[-1])) if len(on) else (0, gray.shape[0] - 1)


def ink_height(gray):
    top, bottom = ink_rows(gray)
    return bottom - top + 1


# ---- our own photos: every detected line >= MIN_TRAIN_CROP_H px tall, from the TRAINING boards only ----
board_cache = OUT_DIR / "board_train_crops.npz"
cache_key = f"{MIN_TRAIN_CROP_H}|{DET_ARGS}|{','.join(train_photos)}"
z = np.load(board_cache, allow_pickle=True) if board_cache.exists() else None
if z is not None and str(z["key"]) == cache_key:
    board_crops, board_src = list(z["crops"]), [str(s) for s in z["src"]]
    print(f"loaded {len(board_crops)} line crops from {board_cache.name} (delete the file to detect again)")
else:
    board_crops, board_src = [], []
    for name in train_photos:
        t0 = time.time()
        img = load(name)
        kept = 0
        for p in detect_full(img):
            if line_height(p) >= 0.8 * MIN_TRAIN_CROP_H:                 # quick pre-filter; the exact check is on the crop
                c = cv2.cvtColor(quad_crop(img, p), cv2.COLOR_BGR2GRAY)
                if c.shape[0] >= MIN_TRAIN_CROP_H and c.shape[1] >= 2 * c.shape[0]:   # tall enough, line-shaped
                    board_crops.append(c); board_src.append(name); kept += 1
        print(f"{name}: {kept} line crops ({time.time() - t0:.0f} s)")
    crops_arr = np.empty(len(board_crops), dtype=object)
    crops_arr[:] = board_crops
    np.savez(board_cache, crops=crops_arr, src=np.array(board_src), key=np.array(cache_key))
assert board_crops, "no line on the training photos is tall enough: lower MIN_TRAIN_CROP_H"
assert not any(s.startswith(TEST_PREFIXES) for s in board_src), "a test board leaked into training"

# hold out 10% of the crops to pick the best epoch. (Not whole photos: most usable lines come from the two
# close-up lecture photos, and the real, independent test is the other boards in part 4 anyway.)
order = np.random.default_rng(SEED).permutation(len(board_crops))
n_val = max(1, len(order) // 10)
board_val = [board_crops[i] for i in order[:n_val]]
board_train = [board_crops[i] for i in order[n_val:]]
assert board_train, "not enough crops for a training/validation split"
INK_RATIO = float(np.median([ink_height(c) / c.shape[0] for c in board_crops]))
print("training crops per photo:", pd.Series(board_src).value_counts().to_dict())
print(f"our photos: {len(board_train)} training crops, {len(board_val)} validation crops, "
      f"heights {min(c.shape[0] for c in board_crops)}-{max(c.shape[0] for c in board_crops)} px; "
      f"ink = {INK_RATIO:.0%} of the box height")

loaded 470 line crops from board_train_crops.npz (delete the file to detect again)
training crops per photo: {'lecture_01.png': 66, 'lecture_11.png': 52, 'lecture_10.png': 51, 'lecture_09.png': 50, 'lecture_13.png': 50, 'lecture_12.png': 47, 'lecture_02.png': 31, 'lecture_05.png': 23, 'lecture_07.png': 23, 'lecture_04.png': 22, 'lecture_06.png': 22, 'lecture_08.png': 22, 'lecture_03.png': 10, 'animal_01.png': 1}
our photos: 423 training crops, 47 validation crops, heights 72-193 px; ink = 47% of the box height


In [ ]:
# ---- IAM lines: pad to the same ink share as our boxes, store IAM_LINE_H px tall ----
def iam_lines(split, n, seed):
    entries = [l.split("\t")[0] for l in (IAM_DIR / f"{split}.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
    pick = np.random.default_rng(seed).choice(len(entries), min(n, len(entries)), replace=False)
    out, before = [], []
    for i in pick:
        g = cv2.imread(str(IAM_DIR / entries[i]), cv2.IMREAD_GRAYSCALE)
        if g is None:
            continue
        top, bottom = ink_rows(g)
        ink = bottom - top + 1
        before.append(ink / g.shape[0])
        box = int(round(ink / INK_RATIO))          # line height at which the ink share equals our boxes'
        if box >= g.shape[0]:                       # ink fills too much of the line: pad with background
            extra = box - g.shape[0]
            g = cv2.copyMakeBorder(g, extra // 2, extra - extra // 2, 0, 0, cv2.BORDER_CONSTANT, value=int(np.median(g)))
        else:                                       # too much empty space above/below: crop around the ink
            y0 = int(np.clip((top + bottom) / 2 - box / 2, 0, g.shape[0] - box))
            g = g[y0:y0 + box]
        out.append(cv2.resize(g, (int(round(g.shape[1] * IAM_LINE_H / g.shape[0])), IAM_LINE_H), interpolation=cv2.INTER_AREA))
    return out, float(np.median(before))


t0 = time.time()
iam_train, iam_ratio = iam_lines("train", IAM_TRAIN_LINES, SEED)
iam_val, _ = iam_lines("val", IAM_VAL_LINES, SEED + 1)
print(f"IAM: {len(iam_train)} training lines, {len(iam_val)} validation lines ({time.time() - t0:.0f} s); "
      f"ink was {iam_ratio:.0%} of the line height, now {np.median([ink_height(g) / g.shape[0] for g in iam_train[:200]]):.0%} "
      f"(our boxes: {INK_RATIO:.0%})")

aug_policy = load_policy(REPO / "augmentation" / "augmentation_policy.yaml")
hr_policy = dict(aug_policy, groups={k: v for k, v in aug_policy["groups"].items() if k in ("geometry", "lighting")})


def has_ink(patch):
    g = patch.astype(np.float32)
    return (g < np.median(g) - 40).mean() > 0.03


def make_pair(src, rng, policy=None):
    """src: grayscale line (uint8). Returns (lr 16x16, hr 64x64) floats in [0, 1], or None."""
    h_lr = int(rng.integers(LR_BOX_RANGE[0], LR_BOX_RANGE[1] + 1))
    H = min(h_lr * SCALE, int(src.shape[0] * MAX_TARGET_FRACTION) // SCALE * SCALE)   # never enlarge: targets = real detail
    if H < HR_PATCH:
        return None
    s = H / src.shape[0]
    need = int(np.ceil((HR_PATCH + 2 * MARGIN + 16) / s))          # source width for one patch + margins
    if src.shape[1] < need:
        return None
    x0 = int(rng.integers(0, src.shape[1] - need + 1))
    win = cv2.resize(src[:, x0:x0 + need], (int(round(need * s)), H), interpolation=cv2.INTER_AREA)
    if policy is not None:
        win = cv2.cvtColor(augment(win, policy, rng)[0], cv2.COLOR_RGB2GRAY)
    Hh, Ww = win.shape[0] // SCALE * SCALE, win.shape[1] // SCALE * SCALE
    win = win[:Hh, :Ww]
    if Hh < HR_PATCH or Ww < HR_PATCH + 2 * MARGIN:
        return None
    for _ in range(8):
        x = int(rng.integers(MARGIN // SCALE, (Ww - HR_PATCH - MARGIN) // SCALE + 1)) * SCALE
        y = int(rng.integers(0, (Hh - HR_PATCH) // SCALE + 1)) * SCALE
        if has_ink(win[y:y + HR_PATCH, x:x + HR_PATCH]):
            break
    else:
        return None
    hr = win[y:y + HR_PATCH, x:x + HR_PATCH]
    strip = win[:, x - MARGIN:x + HR_PATCH + MARGIN]                # full line height, margin left and right
    small = degrade(cv2.cvtColor(strip, cv2.COLOR_GRAY2BGR), Hh // SCALE)
    small_y = cv2.cvtColor(small, cv2.COLOR_BGR2YCrCb)[:, :, 0]
    p, m = HR_PATCH // SCALE, MARGIN // SCALE
    lr = small_y[y // SCALE:y // SCALE + p, m:m + p]
    if lr.shape != (p, p):
        return None
    return lr.astype(np.float32) / 255, hr.astype(np.float32) / 255


def make_set(pool, n, rng, policy=None):
    lrs, hrs, tries = [], [], 0
    while len(lrs) < n:
        tries += 1
        assert tries < 50 * n + 1000, "could not make training pairs: the source lines are too small or too short"
        pair = make_pair(pool[int(rng.integers(len(pool)))], rng, policy)
        if pair is not None:
            lrs.append(pair[0]); hrs.append(pair[1])
    return np.stack(lrs)[:, None], np.stack(hrs)[:, None]            # (n, 1, 16, 16), (n, 1, 64, 64)


# fixed validation pairs (no augmentation), the same for every epoch
VAL = {"iam": make_set(iam_val, VAL_PATCHES, np.random.default_rng(123)),
       "board": make_set(board_val, VAL_PATCHES, np.random.default_rng(124))}
print("validation pairs:", {k: v[0].shape[0] for k, v in VAL.items()})

IAM: 1500 training lines, 200 validation lines (3 s); ink was 37% of the line height, now 47% (our boxes: 47%)
validation pairs: {'iam': 512, 'board': 512}


In [ ]:
# what the CNN learns from: input (16x16, as small as the test lines), bicubic x4, and the sharp target
show = {"IAM (augmented)": make_set(iam_train, 4, np.random.default_rng(7), hr_policy),
        "our photos": make_set(board_train, 4, np.random.default_rng(8))}
fig, axes = plt.subplots(2 * len(show), 4, figsize=(9, 4.6 * len(show)))
for b, (title, (lr_s, hr_s)) in enumerate(show.items()):
    for j in range(4):
        bic = cv2.resize(lr_s[j, 0], (HR_PATCH, HR_PATCH), interpolation=cv2.INTER_CUBIC)
        axes[2 * b, j].imshow(np.hstack([cv2.resize(lr_s[j, 0], (HR_PATCH, HR_PATCH), interpolation=cv2.INTER_NEAREST), bic]),
                              cmap="gray", vmin=0, vmax=1)
        axes[2 * b, j].set_title(f"{title}: input | bicubic", fontsize=8)
        axes[2 * b + 1, j].imshow(hr_s[j, 0], cmap="gray", vmin=0, vmax=1)
        axes[2 * b + 1, j].set_title("target (sharp)", fontsize=8)
for a in axes.ravel():
    a.axis("off")
plt.tight_layout(); plt.show()

<Figure size 900x920 with 16 Axes>

## 3. Training

For each epoch: `STEPS_PER_EPOCH` batches of fresh random pairs -> ESPCN -> **L1 loss** against the sharp target -> backprop -> Adam step.
After each epoch we score the fixed validation pairs and keep the best epoch.

**How long to train?** Not a fixed number of epochs: we train **until the validation score stops improving**.
- No gain (more than `MIN_DELTA` dB) for `LR_PATIENCE` epochs -> halve the learning rate. Smaller steps let it settle into a better spot instead of bouncing around one.
- No gain for `STOP_PATIENCE` epochs -> stop, and keep the best epoch (*early stopping*).
- If it is still improving at `MAX_EPOCHS`, it prints a warning. For reference: bicubic and the untrained (pretrained) ESPCN on the same validation pairs.
PSNR = how close the output is to the sharp target in pixel terms (higher = closer). It is only a training check: the real judge is CER in part 4.

In [ ]:
def psnr(pred, target):
    mse = float(np.mean((np.clip(pred, 0, 1) - target) ** 2))
    return 10 * np.log10(1 / max(mse, 1e-10))


@torch.no_grad()
def predict(model, lr_batch):
    outs = []
    for i in range(0, len(lr_batch), 256):
        outs.append(model(torch.from_numpy(lr_batch[i:i + 256]).to(DEVICE)).cpu().numpy())
    return np.concatenate(outs)


def bicubic_x4(lr_batch):
    return np.stack([cv2.resize(x[0], (HR_PATCH, HR_PATCH), interpolation=cv2.INTER_CUBIC) for x in lr_batch])[:, None]


def train(name, pool, val_key, policy=None):
    path = SR_DIR / f"{name}_x{SCALE}.pt"
    meta = path.with_suffix(".json")
    settings = dict(MAX_EPOCHS=MAX_EPOCHS, STEPS_PER_EPOCH=STEPS_PER_EPOCH, BATCH=BATCH, LEARNING_RATE=LEARNING_RATE,
                    MIN_DELTA=MIN_DELTA, LR_PATIENCE=LR_PATIENCE, STOP_PATIENCE=STOP_PATIENCE, INK_RATIO=round(INK_RATIO, 3),
                    LR_BOX_RANGE=list(LR_BOX_RANGE), n_lines=len(pool), SEED=SEED)
    saved = json.loads(meta.read_text()) if meta.exists() else None
    if path.exists() and not RETRAIN and saved and saved.get("settings") == settings:
        model = ESPCN()
        model.load_state_dict(torch.load(path, map_location="cpu"))
        print(f"[{name}] loaded {path.name} (epoch {saved['best_epoch']}, val PSNR {saved['best_psnr']:.2f} dB); "
              "set RETRAIN = True to train again")
        return model.eval(), []
    if path.exists() and not RETRAIN:
        print(f"[{name}] the saved model was trained with other settings: training again")
    rng = np.random.default_rng(SEED)
    torch.manual_seed(SEED)
    model = load_pretrained(ESPCN()).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    l1 = nn.L1Loss()
    val_lr, val_hr = VAL[val_key]
    model.eval()
    start = psnr(predict(model, val_lr), val_hr)
    best = (start, copy.deepcopy(model.state_dict()), 0)
    hist = [dict(model=name, epoch=0, train_l1=np.nan, val_psnr=start)]
    print(f"[{name}] before training: val PSNR {start:.2f} dB (bicubic {psnr(bicubic_x4(val_lr), val_hr):.2f} dB)")
    bad = 0                                       # epochs since the last real improvement
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); t0 = time.time(); total = 0.0
        for _ in range(STEPS_PER_EPOCH):
            lr_np, hr_np = make_set(pool, BATCH, rng, policy)
            pred = model(torch.from_numpy(lr_np).to(DEVICE))
            loss = l1(pred, torch.from_numpy(hr_np).to(DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        model.eval()
        v = psnr(predict(model, val_lr), val_hr)
        lr_now = opt.param_groups[0]["lr"]
        hist.append(dict(model=name, epoch=ep, train_l1=total / STEPS_PER_EPOCH, val_psnr=v, lr=lr_now))
        note = ""
        if v > best[0] + MIN_DELTA:
            best, bad = (v, copy.deepcopy(model.state_dict()), ep), 0
            note = "  * best so far"
        else:
            bad += 1
            if bad % LR_PATIENCE == 0:
                for g in opt.param_groups:
                    g["lr"] *= 0.5
                note = f"  (no gain for {bad} epochs: learning rate -> {opt.param_groups[0]['lr']:.1e})"
        print(f"[{name}] epoch {ep:>2}: train L1 {total / STEPS_PER_EPOCH:.4f} | val PSNR {v:.2f} dB | "
              f"lr {lr_now:.1e} | {time.time() - t0:.0f} s{note}")
        if bad >= STOP_PATIENCE:
            print(f"[{name}] no improvement for {STOP_PATIENCE} epochs: stopping")
            break
    else:
        print(f"[{name}] WARNING: still improving at MAX_EPOCHS = {MAX_EPOCHS}; raise MAX_EPOCHS for a fully trained model")
    model.load_state_dict(best[1])
    torch.save({k: v.cpu() for k, v in best[1].items()}, path)
    meta.write_text(json.dumps(dict(settings=settings, best_epoch=best[2], best_psnr=round(best[0], 3)), indent=1))
    print(f"[{name}] kept epoch {best[2]} (val PSNR {best[0]:.2f} dB) -> models/superres/{path.name}")
    return model.to("cpu").eval(), hist


history = []
for name, pool, val_key, policy in [("espcn-iam", iam_train, "iam", hr_policy),
                                    ("espcn-board", board_train, "board", None)]:
    model, hist = train(name, pool, val_key, policy)
    TRAINED[name] = model
    history += hist
history = pd.DataFrame(history)

[espcn-iam] the saved model was trained with other settings: training again
[espcn-iam] before training: val PSNR 20.65 dB (bicubic 20.99 dB)
[espcn-iam] epoch  1: train L1 0.0397 | val PSNR 23.39 dB | lr 1.0e-04 | 12 s  * best so far
[espcn-iam] epoch  2: train L1 0.0342 | val PSNR 23.80 dB | lr 1.0e-04 | 10 s  * best so far
[espcn-iam] epoch  3: train L1 0.0326 | val PSNR 24.02 dB | lr 1.0e-04 | 10 s  * best so far
[espcn-iam] epoch  4: train L1 0.0316 | val PSNR 24.18 dB | lr 1.0e-04 | 10 s  * best so far
[espcn-iam] epoch  5: train L1 0.0310 | val PSNR 24.28 dB | lr 1.0e-04 | 11 s  * best so far
[espcn-iam] epoch  6: train L1 0.0305 | val PSNR 24.38 dB | lr 1.0e-04 | 10 s  * best so far
[espcn-iam] epoch  7: train L1 0.0302 | val PSNR 24.43 dB | lr 1.0e-04 | 10 s  * best so far
[espcn-iam] epoch  8: train L1 0.0298 | val PSNR 24.48 dB | lr 1.0e-04 | 11 s  * best so far
[espcn-iam] epoch  9: train L1 0.0297 | val PSNR 24.53 dB | lr 1.0e-04 | 11 s  * best so far
[espcn-iam] epoch 10:

In [ ]:
if history.empty:
    print("models were loaded from disk, so there are no training curves (RETRAIN = True to see them)")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for name, h in history.groupby("model"):
        ax[0].plot(h.epoch, h.train_l1, marker="o", ms=3, label=name)
        line, = ax[1].plot(h.epoch, h.val_psnr, marker="o", ms=3, label=name)
        best_row = h.loc[h.val_psnr.idxmax()]
        ax[1].scatter([best_row.epoch], [best_row.val_psnr], s=120, facecolors="none", edgecolors=line.get_color(), lw=2)
    ax[0].set_title("training L1 loss (lower = closer to sharp)"); ax[0].set_xlabel("epoch")
    ax[1].set_title("validation PSNR, dB (higher = closer); epoch 0 = pretrained, circle = kept"); ax[1].set_xlabel("epoch")
    for a in ax:
        a.grid(alpha=0.3); a.legend()
    plt.tight_layout(); plt.savefig(OUT_DIR / "trained_sr_training_curves.png", dpi=150); plt.show()

# every model on every validation set (does training on one source help on the other?)
rows = []
for key, (val_lr, val_hr) in VAL.items():
    rows.append(dict(validation=key, method="bicubic", psnr=psnr(bicubic_x4(val_lr), val_hr)))
    rows.append(dict(validation=key, method="espcn (pretrained)", psnr=psnr(predict(check_model.to(DEVICE), val_lr), val_hr)))
    for name, m in TRAINED.items():
        rows.append(dict(validation=key, method=name, psnr=psnr(predict(m.to(DEVICE), val_lr), val_hr)))
        m.to("cpu")
check_model.to("cpu")
display(pd.DataFrame(rows).pivot(index="method", columns="validation", values="psnr").round(2))

<Figure size 1200x380 with 2 Axes>

validation,board,iam
method,,
bicubic,19.45,20.99
espcn (pretrained),19.30,20.65
espcn-board,20.57,22.10
espcn-iam,20.19,24.82


In [ ]:
# what the trained models do to a few validation patches from our photos
val_lr, val_hr = VAL["board"]
idx = np.random.default_rng(5).choice(len(val_lr), 4, replace=False)
cols = [("input", lambda x: cv2.resize(x[0, 0], (HR_PATCH, HR_PATCH), interpolation=cv2.INTER_NEAREST)),
        ("bicubic", lambda x: bicubic_x4(x)[0, 0]),
        ("espcn pretrained", lambda x: predict(check_model.to(DEVICE), x)[0, 0])] + \
       [(n, (lambda m: lambda x: predict(m.to(DEVICE), x)[0, 0])(m)) for n, m in TRAINED.items()]
fig, axes = plt.subplots(len(idx), len(cols) + 1, figsize=(2.1 * (len(cols) + 1), 2.2 * len(idx)))
for i, k in enumerate(idx):
    x = val_lr[k:k + 1]
    for j, (title, fn) in enumerate(cols):
        axes[i, j].imshow(np.clip(fn(x), 0, 1), cmap="gray", vmin=0, vmax=1); axes[i, j].set_title(title, fontsize=8)
    axes[i, -1].imshow(val_hr[k, 0], cmap="gray", vmin=0, vmax=1); axes[i, -1].set_title("target", fontsize=8)
for a in axes.ravel():
    a.axis("off")
plt.tight_layout(); plt.show()
for m in list(TRAINED.values()) + [check_model]:
    m.to("cpu")

<Figure size 1260x880 with 24 Axes>

## 4. Evaluation on boards the models never saw (`television`, `cnn`, `quiz`)

**4a. Simulated** (same recipe as part C of the pretrained notebook, but only test boards): clear lines from close-up test photos, shrunk to 16-32 px boxes, enlarged by each method, read. The `none` row will **not** match the pretrained notebook exactly, because the `lecture` lines (now training data) are left out.
**4b. Real far-away photos**: every test photo labelled *Scale and Resolution Variation*. Each ground-truth line is matched with its best reading; results are grouped by the height of the line box the detector found.

In [ ]:
METHODS = ["none", "bicubic+sharpen", "espcn"] + list(TRAINED)
for m in list(TRAINED.values()):
    m.to("cpu").eval()

ref_photos = images.loc[images.image_id.str.startswith(TEST_PREFIXES)
                        & images.condition.isin(["Clean / Control", "Perspective Distortion"]), "image_id"].tolist()
ref = []
for name in ref_photos:
    img, gts = load(name), gt_lines(name)
    polys = detect_full(img)
    crops = [quad_crop(img, p) for p in polys]
    seen = set()
    for crop, (txt, _) in sorted(zip(crops, read(crops)), key=lambda cr: -cr[0].shape[0]):
        if crop.shape[0] < MIN_REF_H or not gts:
            continue
        g = min(gts, key=lambda g: cer(g, txt))
        if cer(g, txt) <= MAX_REF_CER and g not in seen:
            seen.add(g)
            ref.append(dict(photo=name, gt=g, crop=crop))
assert ref, "no reference lines on the test boards"
if len(ref) > MAX_LINES:
    ref = [ref[i] for i in sorted(np.random.default_rng(SEED).choice(len(ref), MAX_LINES, replace=False))]
print(f"{len(ref)} reference lines from {len(set(r['photo'] for r in ref))} test photos "
      f"({len(set(r['gt'] for r in ref))} different texts)")

rows = []
for h in HEIGHTS:
    lrs = [degrade(r["crop"], h) for r in ref]
    for m in METHODS:
        ins = lrs if m == "none" else [enlarge(l, min(TARGET_H / l.shape[0], MAX_SCALE), m) for l in lrs]
        for r, (txt, _) in zip(ref, read(ins)):
            rows.append(dict(height=h, method=m, gt=r["gt"], read=txt, cer=cer(r["gt"], txt)))
sim = pd.DataFrame(rows)
sim.to_csv(OUT_DIR / "trained_sr_simulated.csv", index=False)
sim_table = sim.pivot_table(index="method", columns="height", values="cer", aggfunc="mean").loc[METHODS]
print("SIMULATED, test boards only: mean CER (rows = method, columns = line box height px)")
display(sim_table.round(3))

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizin

48 reference lines from 20 test photos (14 different texts)
SIMULATED, test boards only: mean CER (rows = method, columns = line box height px)


height,16,20,24,28,32
method,,,,,
none,0.686,0.517,0.326,0.230,0.162
bicubic+sharpen,0.765,0.525,0.341,0.240,0.172
espcn,0.694,0.521,0.339,0.232,0.149
espcn-iam,0.835,0.645,0.475,0.339,0.230
espcn-board,0.753,0.591,0.390,0.289,0.195


In [ ]:
far_photos = images.loc[images.image_id.str.startswith(TEST_PREFIXES)
                        & (images.condition == "Scale and Resolution Variation"), "image_id"].tolist()
rows = []
for k, name in enumerate(far_photos, 1):
    t0 = time.time()
    img, gts = load(name), gt_lines(name)
    polys = detect_full(img)
    crops = [quad_crop(img, p) for p in polys]
    reads = {}
    for m in METHODS:
        ins = crops if m == "none" else [enlarge(c, float(np.clip(TARGET_H / c.shape[0], 1, MAX_SCALE)), m) for c in crops]
        reads[m] = [t for t, _ in read(ins)]
    for g in gts:
        if not crops:
            band_h = 0
        else:   # the line's size = box of its best "none" reading, so every method is grouped the same way
            band_h = crops[min(range(len(crops)), key=lambda i: cer(g, reads["none"][i]))].shape[0]
        for m in METHODS:
            txt = min(reads[m], key=lambda t: cer(g, t), default="")
            rows.append(dict(photo=name, gt=g, method=m, box_h=band_h, read=txt, cer=cer(g, txt)))
    print(f"[{k}/{len(far_photos)}] {name}: {len(crops)} boxes, {len(gts)} lines ({time.time() - t0:.0f} s)")
real = pd.DataFrame(rows)
real["band"] = pd.cut(real.box_h, [b[0] for b in BANDS] + [BANDS[-1][1]], right=False,
                      labels=[f"{a}-{b}" if b < 10_000 else f">={a}" for a, b in BANDS])
real.to_csv(OUT_DIR / "trained_sr_real_far.csv", index=False)
real_table = (real.pivot_table(index="method", columns="band", values="cer", aggfunc="mean", observed=False)
              .reindex(index=METHODS, columns=real.band.cat.categories))
counts = real[real.method == "none"].groupby("band", observed=False).size()
print("REAL far-away photos, test boards only: mean CER by line-box height (px)")
display(real_table.round(3))
print("lines per band:", counts.to_dict(), "| overall mean CER:", real.groupby("method").cer.mean().loc[METHODS].round(3).to_dict())

Resized image size (3024x4032) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[1/42] television_01.png: 17 boxes, 2 lines (9 s)


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[2/42] television_02.png: 12 boxes, 2 lines (8 s)


Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[3/42] television_11.png: 29 boxes, 2 lines (11 s)


... (74 outputs left out)

Resized image size (4032x3024) exceeds max_side_limit of 4000. Resizing to fit within limit.


[41/42] quiz_11.png: 19 boxes, 6 lines (9 s)
[42/42] quiz_26.png: 19 boxes, 6 lines (9 s)
REAL far-away photos, test boards only: mean CER by line-box height (px)


,0-16,16-20,20-24,24-32,>=32
method,,,,,
none,NaN,0.569,0.444,0.378,0.225
bicubic+sharpen,NaN,0.619,0.453,0.382,0.233
espcn,NaN,0.648,0.502,0.395,0.223
espcn-iam,NaN,0.625,0.519,0.420,0.259
espcn-board,NaN,0.596,0.451,0.365,0.237


lines per band: {'0-16': 0, '16-20': 9, '20-24': 18, '24-32': 98, '>=32': 139} | overall mean CER: {'none': 0.309, 'bicubic+sharpen': 0.316, 'espcn': 0.32, 'espcn-iam': 0.349, 'espcn-board': 0.311}


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for m in METHODS:
    style = dict(lw=2.5) if m in ("none",) + tuple(TRAINED) else dict(lw=1.2, ls="--")
    ax[0].plot(sim_table.columns, sim_table.loc[m], marker="o", label=m, **style)
    ax[1].plot(range(len(real_table.columns)), real_table.loc[m], marker="o", label=m, **style)
ax[0].set_xlabel("line box height after shrinking (px)"); ax[0].set_title(f"simulated: {len(ref)} lines from test boards")
ax[1].set_xticks(range(len(real_table.columns)))
ax[1].set_xticklabels([f"{b}\n({counts.get(b, 0)} lines)" for b in real_table.columns])
ax[1].set_xlabel("line box height in the real photo (px)"); ax[1].set_title(f"real far-away photos: {len(far_photos)} test photos")
for a in ax:
    a.set_ylabel("mean CER (lower = better)"); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "trained_sr_comparison.png", dpi=150); plt.show()

<Figure size 1400x450 with 2 Axes>

### 4c. Verdict against the rule we set before training

In [ ]:
def mean_cer(lo, hi):
    part = real[(real.box_h >= lo) & (real.box_h < hi)]
    return part.groupby("method").cer.mean(), len(part) // len(METHODS)


small_cer, n_small = mean_cer(16, 24)
big_cer, n_big = mean_cer(24, 10_000)
print(f"lines with 16-24 px boxes: {n_small} | lines with >= 24 px boxes: {n_big}")
if n_small == 0:
    print("no test line falls in 16-24 px, so the rule cannot be judged: add more far-away test photos")
for m in TRAINED:
    if n_small == 0:
        break
    gain_small = float(small_cer["none"] - small_cer[m])
    loss_big = float(big_cer[m] - big_cer["none"]) if n_big else 0.0
    ok = gain_small >= GAIN_NEEDED and loss_big <= 0.01
    print(f"{m:>12}: CER gain over 'none' at 16-24 px = {gain_small:+.3f} (need >= {GAIN_NEEDED}); "
          f"change at >= 24 px = {loss_big:+.3f} (allowed <= +0.01)  ->  {'WORKS' if ok else 'does not meet the rule'}")

lines with 16-24 px boxes: 27 | lines with >= 24 px boxes: 237
   espcn-iam: CER gain over 'none' at 16-24 px = -0.069 (need >= 0.02); change at >= 24 px = +0.037 (allowed <= +0.01)  ->  does not meet the rule
 espcn-board: CER gain over 'none' at 16-24 px = -0.014 (need >= 0.02); change at >= 24 px = +0.001 (allowed <= +0.01)  ->  does not meet the rule


## 5. How to read this (fill in after running)

- **Training check (part 3)**: did validation PSNR rise above bicubic and above the pretrained start? If not, the model learned nothing useful and part 4 will show it.
- **IAM vs our photos**: compare `espcn-iam` and `espcn-board` in part 4, especially on the *real* photos. If `espcn-board` wins, the kind of image (marker on a board, phone camera) matters more than the amount of data.
- **The rule (4c)** decides whether trained SR goes into the pipeline. If neither model meets it, the honest conclusion is: *training on handwriting improved SR, but the recogniser reading the small line directly is still as good or better; the remaining gap is a reading problem* -> fine-tune the recogniser next, and judge it with this same part 4.
- **Caveats**: the test boards have few different texts (`television`, `cnn`, `quiz`); `espcn-board` learned from one main board (`lecture`), so it has seen one person's handwriting style on that board.

## 6. Train on the accepted close–far pairs

Reads `accepted_manifest.csv`, so all `needs_review` and `rejected` rows are excluded. The network learns a same-size stroke-restoration step: at inference, enlarge the distant crop first, **normalise the board** (`pair_data.normalise_board`), convert it to grey (`pair_data.to_grey3`), then pass it through this model.

Changes after the 17 Oct review (details in `scripts/train_accepted_pairs.py`):
1. **Fresh patches every epoch.** Patch choice was seeded by the sample index only, so every epoch re-used the same 1,200 patches.
2. **Board normalisation.** The close and the distant photo of a pair have different exposure and white balance; input and target are each divided by their own board background, so the loss measures strokes, not tone.
3. **Patches on ink, never on warp fill.** 75% of patches are centred on a stroke; the constant-colour fill outside the warped photo is masked out of the loss. Validation uses 8 fixed ink-centred patches per pair (real pairs only) and reports the gain over the untouched input.
4. **Whole input photos held out.** Test = photos 4748 (far), 4750 (side), 4697 (medium); validation = 4736, 4705, 4673. No test photo's blur, noise or exposure is seen in training.
6. **Greyscale (run 2).** Run 1 in colour mostly learned to *recolour* the 17 Oct blue marker (dark and dull from far away, bright blue close up): +0.9 dB on that board, −0.2 dB on other boards, and no OCR benefit. Input and target are now normalised and converted to grey, so a gain can only come from the strokes. Run 1 is saved in `outputs/_sr_review/run1_colour_model/`.
7. **Ink-depth matching + other boards (run 3).** Run 2 (grey) still learned to *lighten* the 17 Oct strokes (the marker looks darker from far away). Each target's ink depth is now matched to its input's (`pair_data.match_ink_depth`), so only stroke shape and sharpness are left to learn. Training also gets 276 synthetic pairs from the quiz, lecture-notes and animal boards (`scripts/make_airdrop_synthetic_pairs.py`; black and blue markers, other handwriting); the television and CNN boards stay unseen for the test. Run 2 is saved in `outputs/_sr_review/run2_grey_model/`.
8. **Validation on another board (run 4).** The 17 Oct validation photos are already nearly sharp, so on their own they could not show sharpening, and early stopping kept an almost untrained model (run 3 saved epoch 1, yet it already gave +0.35 dB on other boards). Real pairs of the **television board** now join validation, each validation set counts equally, and patience is 10. The **CNN board** stays unseen as the cross-board test. Run 3 is saved in `outputs/_sr_review/run3_ink_matched/`.
9. **A cross-board test.** `scripts/make_airdrop_test_pairs.py` builds real far→sharp pairs from the Airdrop line crops of two other boards (television, CNN) with human transcriptions.

10. **Stroke-aware loss (M1).** Run 4 raised PSNR on the unseen CNN board (+1.36 dB, 24/24 pairs) but not PaddleOCR's reading (CER 0.169 → 0.174; 3 better / 16 same / 5 worse): a pixel loss is dominated by board pixels and rewards an averaged soft blob where a stroke is uncertain. M1 keeps everything else from run 4 and changes only the loss: stroke pixels count 5× in the L1 term, plus a soft **Dice** term (missing or extra ink) and a soft **clDice** term (Shit et al., CVPR 2021: compares stroke *skeletons*, so a broken stroke or two strokes merged into a blob is penalised even when few pixels differ). Weights 0.25 and 0.3 make each stroke term about 0.9× the ink-weighted L1 term at the start of training (averaged over the training mix of real pairs; an earlier calibration on the median patch gave clDice 0.5 and was corrected before the full run). The best epoch is still chosen on PSNR gain, so the loss is the only change; validation now also prints the stroke Dice. Run 4 is saved in `outputs/_sr_review/run4_pixel_loss/`.
11. **More test data, from the teammates' PhotosToUse folder** (`scripts/make_phototouse_pairs.py`). The folder has no far shots small enough for a real test (its "far" letters are still 50–250 px tall), so it adds:
    - a **low-resolution test** (synthetic): 15 labelled whiteboard lines from 11 texts, shrunk to the real far-crop range (0.22 and 0.35 source pixels per target pixel) by area averaging only, then enlarged back: 30 pairs;
    - a **camera-shake stress test**: the 8 real camera-shake crops of single-line texts, scaled to the clear line (not pixel-aligned: CER only). Motion blur is not what the model was trained for;
    - 60 synthetic **training** pairs from 15 OnePlus whiteboard photos whose texts are in neither test (`PAIR_USE_PHOTOTOUSE`, off in M1 so the loss is the only change).

12. **Recognition loss (M2).** M1 failed too (CNN board CER 0.171 vs 0.165 without the CNN; low-resolution set worse), and its training log showed the model hardly reducing the stroke terms: a shape loss does not know what a *letter* is. Every text-SR method that does improve reading ties its loss to a text **recogniser** (TextSR's text perceptual loss, TPGSR / TATT text priors, Text Gestalt's stroke attention; see `docs/SUPER_RESOLUTION_RUNS.md`). PaddleOCR cannot give gradients (Paddle, not PyTorch), so a small PyTorch CRNN line recogniser is trained first (cell below: IAM handwritten lines + our whiteboard crops, never the B2 / B3 boards) and then **frozen**. M2 = run 4's loss + that recogniser's CTC loss on two whole restored lines per step; its weight is set automatically so the term starts as large as the pixel loss. PaddleOCR, a *different* recogniser, still judges, so a gain cannot come from fooling the training recogniser.

**Pass/fail rule (fixed before M1 was trained).** A change counts as an improvement only if, on **real pairs from an unseen board**, (1) the mean restored CER is lower than both `input, no CNN` and `raw input`, and (2) more pairs improve than worsen against `input, no CNN`. Today the only such set is the Airdrop CNN board (24 pairs, but only 6 different lines), so any verdict is **preliminary** until the far-shot photo set exists. The low-resolution set is reported but does not decide; the shake set is a stress test.

**Order:** (1) `RUN_TAG = 'run4_pixel_loss'`, `PAIR_TRAIN = False` and run section 6: run 4 scored on the new test sets (the baseline). (2) `RUN_TAG = 'm1_stroke_loss'`, `PAIR_TRAIN = True`, `PAIR_SMOKE = True`: a 2-epoch check (about a minute). (3) `PAIR_SMOKE = False`: train M1, then run the rest of section 6. Every result file is named after `RUN_TAG`, so runs never overwrite each other.

### 6-0. Line recogniser for the recognition loss (M2)

Trains the frozen PyTorch CRNN used by M2 (`scripts/train_line_recognizer.py`, model in `scripts/line_recognizer.py`): 6,482 IAM handwritten lines plus 548 of our whiteboard crops (Airdrop boards B1, B4, B5 and the 17 Oct training photos). The television (B2) and CNN (B3) boards and the 17 Oct validation / test photos are left out. Images go through the same board normalisation as the SR model, so the recogniser reads exactly what the SR model outputs.

The best epoch is chosen on IAM validation lines + 17 Oct validation-photo crops. The CER on the CNN-board crops is printed only as a sanity check (never used to choose). **What to expect:** low CER on IAM validation (roughly 0.1 or below) and a usable CER on the CNN-board crops (well below 0.5). If the CNN-board CER is close to 1, the recogniser cannot read our boards and M2 would be pointless — stop and tell me.

Run once (about 20–40 minutes); `REC_TRAIN = False` afterwards keeps the saved recogniser.

In [27]:
import subprocess
import sys
from pathlib import Path

REC_REPO = next(f for f in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                if (f / 'scripts' / 'train_line_recognizer.py').exists())
REC_TRAIN = True     # False = keep the saved recogniser
REC_SMOKE = False     # True = one tiny epoch, only to check that it runs; then set False for the real training
REC_MODEL = REC_REPO / 'models' / 'superres' / 'line_recognizer_crnn.pt'

rec_script = str(REC_REPO / 'scripts' / 'train_line_recognizer.py')
subprocess.run([sys.executable, rec_script, '--self-test'], cwd=REC_REPO, check=True)   # a few seconds
if REC_TRAIN:
    subprocess.run([sys.executable, rec_script, '--output', str(REC_MODEL)] + (['--smoke'] if REC_SMOKE else []),
                   cwd=REC_REPO, check=True)
print(f'recogniser: {REC_MODEL} (exists: {REC_MODEL.exists()})')

  PASS  encode/decode
  PASS  unknown chars dropped
  PASS  CER
  PASS  output shape (T, B, C)
  PASS  CTC finite, gradient reaches the image
  PASS  numpy and torch preprocessing agree (height 95)
  PASS  numpy and torch preprocessing agree (height 300)
recogniser self-test passed
loading and normalising crops (once) ...
  done in 30s: iam_train 6482, own_train 548, iam_val 400, own_choose 22, own_report 90
charset: 86 characters; characters of held-out labels not in it: 0
device: mps | parameters: 3.8 M


<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/train_line_recognizer.py:244: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:821.)
  losses.append(float(loss))


epoch 01: CTC loss 3.986 | CER IAM val 1.000, 17 Oct val photos 1.000 | (report only) CNN board B3 crops 1.000 * best
epoch 02: CTC loss 1.676 | CER IAM val 0.298, 17 Oct val photos 0.219 | (report only) CNN board B3 crops 0.463 * best
epoch 03: CTC loss 0.791 | CER IAM val 0.150, 17 Oct val photos 0.074 | (report only) CNN board B3 crops 0.341 * best
epoch 04: CTC loss 0.562 | CER IAM val 0.128, 17 Oct val photos 0.079 | (report only) CNN board B3 crops 0.376 * best
epoch 05: CTC loss 0.435 | CER IAM val 0.108, 17 Oct val photos 0.033 | (report only) CNN board B3 crops 0.292 * best
epoch 06: CTC loss 0.365 | CER IAM val 0.089, 17 Oct val photos 0.021 | (report only) CNN board B3 crops 0.325 * best
epoch 07: CTC loss 0.315 | CER IAM val 0.084, 17 Oct val photos 0.025 | (report only) CNN board B3 crops 0.272 * best
epoch 08: CTC loss 0.274 | CER IAM val 0.075, 17 Oct val photos 0.015 | (report only) CNN board B3 crops 0.298 * best
epoch 09: CTC loss 0.238 | CER IAM val 0.071, 17 Oct val

In [30]:
import subprocess
import sys
from pathlib import Path

def find_whiteboard_repo(start=Path.cwd()):
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / 'scripts' / 'train_accepted_pairs.py').exists():
            return folder
    raise FileNotFoundError('Cannot find scripts/train_accepted_pairs.py')

PAIR_REPO = find_whiteboard_repo()

# Each run has a tag; the model file and every result CSV are named after it, so runs never overwrite each other.
RUNS = {
    # run 4: plain L1 + gradient loss. Saved baseline - evaluate it with PAIR_TRAIN = False, never retrain it.
    'run4_pixel_loss': dict(ink_weight=1.0, dice_weight=0.0, cldice_weight=0.0, phototouse=False,
                            model='accepted_pair_restorer.pt'),
    # M1: the same run, only the loss changes (stroke-weighted L1 + Dice + clDice)
    'm1_stroke_loss': dict(ink_weight=5.0, dice_weight=0.25, cldice_weight=0.3, phototouse=False,
                           model='m1_stroke_loss.pt'),
    # next run, if M1 helps: M1 + the 60 PhotosToUse training pairs (one change per run)
    'm1_plus_phototouse': dict(ink_weight=5.0, dice_weight=0.25, cldice_weight=0.3, phototouse=True,
                               model='m1_plus_phototouse.pt'),
    # M2: run 4's loss + the frozen line recogniser's CTC loss (train the recogniser in the cell above first)
    'm2_recognition_loss': dict(ink_weight=1.0, dice_weight=0.0, cldice_weight=0.0, phototouse=False,
                                rec_ratio=1.0, model='m2_recognition_loss.pt'),
}
RUN_TAG = 'm2_recognition_loss'   # 'run4_pixel_loss' / 'm1_stroke_loss' = re-score a saved run with PAIR_TRAIN = False
PAIR_TRAIN = True             # False = skip training and evaluate the saved model of RUN_TAG (e.g. after a kernel restart)
PAIR_SMOKE = False             # True = 2 tiny epochs, only to check that the script runs; then False for the real run

PAIR_EPOCHS = 30
PAIR_STEPS_PER_EPOCH = 150
PAIR_BATCH_SIZE = 8           # patches per step (2 pairs x 4 patches)
PAIR_PATCH_SIZE = 128
PAIR_MIN_SPP = 0.0            # e.g. 0.25 = train only on pairs that needed at most 4x enlargement
PAIR_USE_OTHER_BOARDS = True  # run 3: add synthetic pairs from other Airdrop boards (not the test boards)
PAIR_VAL_BOARDS = ['airdrop_B2']   # run 4: real pairs of the television board join validation; CNN board stays test-only
PAIR_PATIENCE = 10            # early stopping: epochs without improvement (validation is small and noisy)
PAIR_SELECT_ON = 'psnr'       # best epoch chosen on PSNR gain (as in run 4); 'dice' = on stroke-Dice gain

run = RUNS[RUN_TAG]
PAIR_MODEL = PAIR_REPO / 'models' / 'superres' / run['model']
airdrop_pairs = PAIR_REPO / 'outputs' / 'paired_dataset_airdrop' / 'airdrop_test_manifest.csv'
synth_manifest = PAIR_REPO / 'outputs' / 'paired_dataset_airdrop_synth' / 'airdrop_synth_manifest.csv'
phototouse = PAIR_REPO / 'outputs' / 'paired_dataset_phototouse'
phototouse_train = phototouse / 'train_synth' / 'phototouse_train_manifest.csv'
if PAIR_USE_OTHER_BOARDS and not synth_manifest.exists():
    subprocess.run([sys.executable, str(PAIR_REPO / 'scripts' / 'make_airdrop_synthetic_pairs.py')], cwd=PAIR_REPO, check=True)
if not phototouse_train.exists():   # also builds the low-resolution and camera-shake test sets
    subprocess.run([sys.executable, str(PAIR_REPO / 'scripts' / 'make_phototouse_pairs.py')], cwd=PAIR_REPO, check=True)

print(f"Run: {RUN_TAG} -> {PAIR_MODEL.name}")
if PAIR_TRAIN:
    if RUN_TAG == 'run4_pixel_loss':
        raise ValueError('run 4 is the saved baseline: set PAIR_TRAIN = False to evaluate it (retraining would overwrite it)')
    trainer = str(PAIR_REPO / 'scripts' / 'train_accepted_pairs.py')
    subprocess.run([sys.executable, trainer, '--self-test'], cwd=PAIR_REPO, check=True)   # loss functions, a few seconds
    extra = [str(synth_manifest)] if PAIR_USE_OTHER_BOARDS else []
    extra += [str(phototouse_train)] if run['phototouse'] else []
    rec_args = (['--recognizer', str(PAIR_REPO / 'models' / 'superres' / 'line_recognizer_crnn.pt'),
                 '--rec-ratio', str(run['rec_ratio'])] if run.get('rec_ratio') else [])
    command = [
        sys.executable, trainer,
        '--output', str(PAIR_MODEL),
        '--epochs', str(PAIR_EPOCHS),
        '--steps-per-epoch', str(PAIR_STEPS_PER_EPOCH),
        '--batch-size', str(PAIR_BATCH_SIZE),
        '--patch-size', str(PAIR_PATCH_SIZE),
        '--min-spp', str(PAIR_MIN_SPP),
        '--patience', str(PAIR_PATIENCE),
        '--ink-weight', str(run['ink_weight']),
        '--dice-weight', str(run['dice_weight']),
        '--cldice-weight', str(run['cldice_weight']),
        '--select-on', PAIR_SELECT_ON,
    ] + rec_args + (['--smoke'] if PAIR_SMOKE else []) + (['--extra-manifest', *extra] if extra else []) \
      + (['--extra-val-manifest', str(airdrop_pairs), '--extra-val-groups', *PAIR_VAL_BOARDS] if PAIR_VAL_BOARDS else [])
    print('Training from accepted_manifest.csv' + (' + extra: ' + ', '.join(Path(e).name for e in extra) if extra else '') + ' ...')
    subprocess.run(command, cwd=PAIR_REPO, check=True)
else:
    print('PAIR_TRAIN = False: using the saved model')

Run: m2_recognition_loss -> m2_recognition_loss.pt
  PASS  identical -> Dice loss ~0
  PASS  identical -> clDice loss ~0
  PASS  gap raises clDice
  PASS  merge raises clDice
  PASS  ink weight 1 == masked L1
  PASS  ink weight 5 > masked L1 for a blurred stroke
  PASS  blank patch -> no stroke loss
  PASS  gradients flow and are finite
  PASS  recognition loss finite, reaches the SR model, recogniser stays frozen
  PASS  recogniser CER of input and output computed
values: Dice blurred 0.167 | clDice blurred 0.000, broken 0.066, merged 0.042
self-test passed
Training from accepted_manifest.csv + extra: airdrop_synth_manifest.csv ...
extra training pairs: 276 from airdrop_synth_manifest.csv (groups: ['airdrop_synth_B1', 'airdrop_synth_B4', 'airdrop_synth_B5'])
train: 312 pairs (174 real, 138 synthetic) from 11 input photos  [17 Oct manifest]
  val:  20 pairs (20 real, 0 synthetic) from 3 input photos  [17 Oct manifest]
 test:  40 pairs (40 real, 0 synthetic) from 3 input photos  [17 Oct

In [31]:
import pandas as pd
import torch

split_path = PAIR_REPO / 'outputs' / 'paired_dataset_17oct' / 'accepted_split_manifest.csv'
model_path = PAIR_MODEL
split_rows = pd.read_csv(split_path, keep_default_na=False)
checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

display(split_rows.groupby(['split', 'pair_type', 'input_role']).size().rename('pairs').reset_index())
for split in ('val', 'test'):
    photos = sorted(split_rows.loc[(split_rows.split == split) & (split_rows.pair_type == 'real'), 'input_photo'].unique())
    print(f'{split} input photos (never seen in training): {photos}')
print(f"Run {RUN_TAG}: best epoch {checkpoint['epoch']}, loss {checkpoint.get('loss', 'L1 + 0.2 x gradient (run 4)')}, "
      f"best epoch chosen on {checkpoint.get('select_on', 'psnr')}")
for name, v in checkpoint.get('val_sets', {}).items():
    print(f"Validation set {name}: stroke-patch PSNR {v['psnr']:.2f} dB "
          f"(untouched input {v['input_psnr']:.2f} dB, gain {v['psnr'] - v['input_psnr']:+.2f} dB)"
          + (f"; stroke Dice {v['dice']:.3f} (input {v['input_dice']:.3f})" if 'dice' in v else ''))
print(f"Mean validation gain (every set counts equally): {checkpoint['val_psnr'] - checkpoint['val_input_psnr']:+.2f} dB")
recog = checkpoint.get('val_recog_cer')
if recog and recog.get('lines'):
    print(f"Training recogniser CER on {recog['lines']} labelled validation lines: input {recog['input']:.3f} -> restored {recog['restored']:.3f}")
print(f"Saved model: {model_path}")

,split,pair_type,input_role,pairs
0,test,real,far_input,18
1,test,real,medium_input,5
2,test,real,side_angle_input,17
3,train,real,far_input,49
4,train,real,medium_input,117
5,train,real,side_angle_input,8
6,train,synthetic,synthetic_from_close_target,138
7,val,real,far_input,6
8,val,real,medium_input,6
9,val,real,side_angle_input,8


val input photos (never seen in training): ['IMG_4673.HEIC', 'IMG_4705.HEIC', 'IMG_4736.HEIC']
test input photos (never seen in training): ['IMG_4697.HEIC', 'IMG_4748.HEIC', 'IMG_4750.HEIC']
Run m2_recognition_loss: best epoch 25, loss {'ink_weight': 1.0, 'dice_weight': 0.0, 'cldice_weight': 0.0, 'skeleton_iterations': 15, 'rec_ratio': 1.0, 'rec_weight': 0.4236511498615852, 'recognizer': '<LOCAL_ASSIGNMENT>/whiteboard-ocr/models/superres/line_recognizer_crnn.pt'}, best epoch chosen on psnr
Validation set 17 Oct: stroke-patch PSNR 14.47 dB (untouched input 14.34 dB, gain +0.13 dB); stroke Dice 0.787 (input 0.786)
Validation set airdrop_B2: stroke-patch PSNR 19.35 dB (untouched input 19.01 dB, gain +0.34 dB); stroke Dice 0.782 (input 0.781)
Mean validation gain (every set counts equally): +0.23 dB
Training recogniser CER on 10 labelled validation lines: input 0.000 -> restored 0.000
Saved model: <LOCAL_ASSIGNMENT>/whiteboard-ocr/models/superres/m2_recognition_loss.pt


### 6a. Compare the trained model with the original test input — whole canvases

Each test canvas is restored **whole, at its native scale** (tiles with context), and compared with the clear target in the normalised grey space (target ink depth matched to the input, as in training, so PSNR measures stroke shape and sharpness), over all real pixels and over the **stroke area** (target ink, slightly grown; warp fill excluded). Only real pairs are scored. Two test sets:
- **17 Oct, held-out photos** — same board, but input photos never seen in training;
- **Airdrop, unseen board** — the CNN board: never used in training or validation (the television board is now used for validation, so it is not tested);
- **PhotosToUse, low resolution (synthetic)** — 15 labelled lines from texts never used in training, shrunk to the real far-crop range and enlarged back (resolution loss only).

The camera-shake stress test is **not** scored here: shaken text cannot be pixel-aligned with its clear version, so PSNR would be meaningless. It is scored by CER in 6c.

The model starts as an identity map, so **gain** (restored − input) is the number that matters. Results are also split by view and by how much the input had to be enlarged (`spp_band`).

In [32]:
import sys; sys.modules.pop('pair_data', None)   # drop the copy cached from run 1
import csv
import importlib.util
import subprocess
import sys
import numpy as np
import pandas as pd

trainer_path = PAIR_REPO / 'scripts' / 'train_accepted_pairs.py'
spec = importlib.util.spec_from_file_location('accepted_pair_trainer', trainer_path)
pair_trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pair_trainer)
pair_data = pair_trainer.pair_data

eval_device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
test_model = pair_trainer.load_model(model_path, eval_device)
print(f'Evaluating run {RUN_TAG}: {model_path.name}')

with split_path.open(newline='', encoding='utf-8') as handle:
    split_records = list(csv.DictReader(handle))
test_records = [row for row in split_records if row['split'] == 'test' and row['pair_type'] == 'real']

airdrop_manifest = PAIR_REPO / 'outputs' / 'paired_dataset_airdrop' / 'airdrop_test_manifest.csv'
if not airdrop_manifest.exists():
    subprocess.run([sys.executable, str(PAIR_REPO / 'scripts' / 'make_airdrop_test_pairs.py')], cwd=PAIR_REPO, check=True)
with airdrop_manifest.open(newline='', encoding='utf-8') as handle:
    airdrop_all = [row for row in csv.DictReader(handle) if row['review_status'] == 'accepted']
val_boards = set(PAIR_VAL_BOARDS)
airdrop_records = [row for row in airdrop_all if row['group_id'] not in val_boards]   # validation boards are NOT tested
print(f"Airdrop pairs: {len(airdrop_all)}; used for validation {sorted(val_boards)}; test: {sorted({r['group_id'] for r in airdrop_records})}")

def read_manifest(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return [row for row in csv.DictReader(handle) if row['review_status'] == 'accepted']

lowres_records = read_manifest(phototouse / 'lowres_test' / 'phototouse_lowres_test_manifest.csv')
shake_records = read_manifest(phototouse / 'shake_test' / 'phototouse_shake_test_manifest.csv')
TEST_SETS = [('17 Oct, held-out photos', test_records), ('Airdrop, unseen board', airdrop_records),
             ('PhotosToUse, low resolution (synthetic)', lowres_records),
             ('PhotosToUse, camera shake (stress test)', shake_records)]
NOT_ALIGNED = {'PhotosToUse, camera shake (stress test)'}      # CER only (6c)
frames = []
for set_name, records in TEST_SETS:
    if set_name in NOT_ALIGNED:
        continue
    print(f'Evaluating {len(records)} pairs: {set_name} ...')
    frame = pd.DataFrame(pair_trainer.evaluate_pairs(test_model, records, eval_device))
    frame.insert(0, 'set', set_name)
    frames.append(frame)
psnr_results = pd.concat(frames, ignore_index=True)
psnr_path = PAIR_REPO / 'outputs' / 'paired_dataset_17oct' / f'{RUN_TAG}_psnr_full_canvas.csv'
psnr_results.to_csv(psnr_path, index=False)

def summarise(frame, by):
    grouped = frame.groupby(by)
    return pd.DataFrame({
        'pairs': grouped.size(),
        'input PSNR, strokes (dB)': grouped.psnr_input_ink.mean(),
        'restored PSNR, strokes (dB)': grouped.psnr_restored_ink.mean(),
        'gain, strokes (dB)': grouped.gain_ink.mean(),
        'pairs improved': grouped.gain_ink.apply(lambda s: int((s > 0).sum())),
        'gain, whole canvas (dB)': grouped.gain_all.mean(),
    }).round(2)

display(summarise(psnr_results, 'set'))
display(summarise(psnr_results, ['set', 'input_role']))
display(summarise(psnr_results, ['set', 'spp_band']))
print(f'Saved per-pair results: {psnr_path}')

Evaluating run m2_recognition_loss: m2_recognition_loss.pt
Airdrop pairs: 34; used for validation ['airdrop_B2']; test: ['airdrop_B3']
Evaluating 40 pairs: 17 Oct, held-out photos ...
Evaluating 24 pairs: Airdrop, unseen board ...
Evaluating 30 pairs: PhotosToUse, low resolution (synthetic) ...


,pairs,"input PSNR, strokes (dB)","restored PSNR, strokes (dB)","gain, strokes (dB)",pairs improved,"gain, whole canvas (dB)"
set,,,,,,
"17 Oct, held-out photos",40,11.54,11.61,0.07,27,0.16
"Airdrop, unseen board",24,15.33,15.68,0.35,23,0.40
"PhotosToUse, low resolution (synthetic)",30,20.54,20.04,-0.50,15,-0.61


pairs  \
set                                     input_role                              
17 Oct, held-out photos                 far_input                          18   
                                        medium_input                        5   
                                        side_angle_input                   17   
Airdrop, unseen board                   Perspective Distortion              1   
                                        Scale and Resolution Variation     23   
PhotosToUse, low resolution (synthetic) downscaled_clear                   30   

                                                                        input PSNR, strokes (dB)  \
set                                     input_role                                                 
17 Oct, held-out photos                 far_input                                          12.20   
                                        medium_input                                        9.99   
                                        side_angle_input                                   11.29   
Airdrop, unseen board                   Perspective Distortion                             13.89   
                                        Scale and Resolution Variation                     15.39   
PhotosToUse, low resolution (synthetic) downscaled_clear                                   20.54   

                                                                        restored PSNR, strokes (dB)  \
set                                     input_role                                                    
17 Oct, held-out photos                 far_input                                             12.18   
                                        medium_input                                          10.17   
                                        side_angle_input                                      11.42   
Airdrop, unseen board                   Perspective Distortion                                14.36   
                                        Scale and Resolution Variation                        15.73   
PhotosToUse, low resolution (synthetic) downscaled_clear                                      20.04   

                                                                        gain, strokes (dB)  \
set                                     input_role                                           
17 Oct, held-out photos                 far_input                                    -0.02   
                                        medium_input                                  0.18   
                                        side_angle_input                              0.14   
Airdrop, unseen board                   Perspective Distortion                        0.47   
                                        Scale and Resolution Variation                0.34   
PhotosToUse, low resolution (synthetic) downscaled_clear                             -0.50   

                                                                        pairs improved  \
set                                     input_role                                       
17 Oct, held-out photos                 far_input                                    6   
                                        medium_input                                 5   
                                        side_angle_input                            16   
Airdrop, unseen board                   Perspective Distortion                       1   
                                        Scale and Resolution Variation              22   
PhotosToUse, low resolution (synthetic) downscaled_clear                            15   

                                                                        gain, whole canvas (dB)  
set                                     input_role                                               
17 Oct, held-out photos                 far_input                                          0.10  
                             

pairs  \
set                                     spp_band                      
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)     19   
                                        <0.15 (>6.7x)             1   
                                        >=0.30 (<3.3x)           20   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)      5   
                                        >=0.30 (<3.3x)           19   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)     15   
                                        >=0.30 (<3.3x)           15   

                                                              input PSNR, strokes (dB)  \
set                                     spp_band                                         
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)                     12.07   
                                        <0.15 (>6.7x)                            12.38   
                                        >=0.30 (<3.3x)                           10.99   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)                     14.69   
                                        >=0.30 (<3.3x)                           15.50   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)                     18.61   
                                        >=0.30 (<3.3x)                           22.47   

                                                              restored PSNR, strokes (dB)  \
set                                     spp_band                                            
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)                        12.06   
                                        <0.15 (>6.7x)                               12.30   
                                        >=0.30 (<3.3x)                              11.15   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)                        14.90   
                                        >=0.30 (<3.3x)                              15.88   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)                        18.61   
                                        >=0.30 (<3.3x)                              21.47   

                                                              gain, strokes (dB)  \
set                                     spp_band                                   
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)               -0.01   
                                        <0.15 (>6.7x)                      -0.08   
                                        >=0.30 (<3.3x)                      0.16   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)                0.21   
                                        >=0.30 (<3.3x)                      0.38   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)               -0.01   
                                        >=0.30 (<3.3x)                     -1.00   

                                                              pairs improved  \
set                                     spp_band                               
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)               7   
                                        <0.15 (>6.7x)                      0   
                                        >=0.30 (<3.3x)                    20   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)               4   
                                        >=0.30 (<3.3x)                    19   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)               9   
                                        >=0.30 (<3.3x)                     6   

                                                              gain, whole canvas (dB)  
set                                     spp_band                                       
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)                     0.11  
                               

Saved per-pair results: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/paired_dataset_17oct/m2_recognition_loss_psnr_full_canvas.csv


### 6b. Inspect whether the character strokes remain faithful

The model can increase PSNR while still damaging a letter. These rows show the normalised grey input, the CNN restoration and the clear target on whole held-out canvases (cropped to the writing). The first rows are the largest stroke-area gains, the last row is the weakest case, so the display does not hide failures.

In [33]:
import matplotlib.pyplot as plt   # so section 6 also runs on its own after a kernel restart

def writing_window(images, ink, pad=20, max_ratio=5.0):
    ys, xs = np.nonzero(ink)
    if len(xs) == 0:
        return images
    y0, y1 = max(0, ys.min() - pad), ys.max() + pad
    x0 = max(0, xs.min() - pad)
    x1 = min(xs.max() + pad, x0 + int(max_ratio * (y1 - y0)))
    return [image[y0:y1, x0:x1] for image in images]

records_by_id = {row['pair_id']: row for _, records in TEST_SETS for row in records}
ranked = psnr_results.sort_values('gain_ink')
picks = pd.concat([ranked.tail(3).iloc[::-1], ranked.head(1)])

fig, axes = plt.subplots(len(picks), 3, figsize=(18, 2.8 * len(picks)))
for row_number, (_, result) in enumerate(picks.iterrows()):
    record = records_by_id[result.pair_id]
    inp, tgt, ok = pair_trainer.load_pair(record)
    restored = pair_trainer.restore_full(test_model, inp, eval_device)
    views = writing_window([inp, restored, tgt], pair_data.ink_mask(tgt, grow=0) & ok)
    for column, (title, image) in enumerate(zip(['Input (enlarged, normalised)', 'CNN restoration', 'Clear target'], views)):
        axes[row_number, column].imshow(np.clip(image, 0, 1))
        axes[row_number, column].set_title(title if column else
                                           f'{result.set} | {title} | stroke gain {result.gain_ink:+.2f} dB', fontsize=9)
        axes[row_number, column].axis('off')
plt.tight_layout()
visual_path = PAIR_REPO / 'outputs' / 'paired_dataset_17oct' / f'{RUN_TAG}_visual_comparison.png'
plt.savefig(visual_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved visual comparison: {visual_path}')

<Figure size 1800x1120 with 12 Axes>

Saved visual comparison: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/paired_dataset_17oct/m2_recognition_loss_visual_comparison.png


### 6c. Does restoration actually help PaddleOCR?

This is the decision metric for the project. Only real test pairs with a line-level label (`cer_eligible=yes`) are scored. Each canvas is restored **at its native scale** (as the model was trained); only afterwards is every version shrunk the same way for OCR.

Run 1 showed that PaddleOCR reads the *normalised* image worse than the raw photo (Airdrop: worse on 10 of 34 pairs, better on 5), so every processed version is mapped back to the photo's own brightness (`pair_data.denormalise_grey`) before OCR. PaddleOCR reads four versions:
- **raw input** — what the pipeline sees today;
- **input, no CNN** — grey + normalisation, mapped back; exactly the restored path without the network;
- **restored** — the same path with the CNN;
- **clear target** — the real close-up photo: the best restoration could possibly do.

**Colour path (added after run 4's baseline, before M1 was trained).** Scoring run 4 showed that the grey path itself costs accuracy: `input, no CNN` already read worse than `raw input` before the network did anything (Airdrop 0.157 → 0.169, low-resolution set 0.153 → 0.184), because PaddleOCR reads small blue-marker text better in colour. Two more versions are therefore scored, following the original SRCNN recipe: the model's grey result gives the **brightness**, the photo keeps its own **colour** (`pair_data.put_colour_back`, Y channel of YCrCb):
- **input, no CNN, colour** — should read almost exactly like `raw input` (a check that the path itself is harmless);
- **restored, colour** — the CNN's brightness with the photo's colour. **This is the pipeline the pass/fail verdict judges from M1 on**; the grey path is reported next to it.

**The test of the model is `restored` vs `input, no CNN`** (the only difference between them is the CNN), and the pass/fail rule above also requires beating `raw input`. The cell ends with the verdict for the real unseen-board set. Character Error Rate (CER; lower is better). Airdrop labels are human (`Airdrop_Labels.xlsx`); the 17 Oct labels are an unchecked AI first pass, and one 17 Oct pair ("Complex cells respond …") is unreadable even in the clear target, so treat the 17 Oct CER as rough.

In [34]:
import pandas as pd
import cv2
import jiwer
from paddleocr import TextRecognition

OCR_MAX_SIDE = 1600
OCR_MODEL_NAME = 'PP-OCRv6_medium_rec'
accepted_recogniser = TextRecognition(model_name=OCR_MODEL_NAME)
VERSIONS = ['raw input', 'input, no CNN', 'restored', 'input, no CNN, colour', 'restored, colour', 'clear target']

def normalise_text(value):
    return ' '.join(str(value).lower().split())

def character_error_rate(reference, prediction):
    reference = normalise_text(reference)
    prediction = normalise_text(prediction) or ' '
    return min(float(jiwer.cer(reference, prediction)), 1.0)

def shrink_for_ocr(image_bgr, max_side=OCR_MAX_SIDE):
    # applied AFTER restoration, identically to every version
    height, width = image_bgr.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale == 1.0:
        return image_bgr
    return cv2.resize(image_bgr, (max(1, round(width * scale)), max(1, round(height * scale))), interpolation=cv2.INTER_AREA)

def recognise_line(image_bgr):
    result = accepted_recogniser.predict([image_bgr])[0]
    return str(result['rec_text']), float(result['rec_score'])

ocr_rows = []
for set_name, records in TEST_SETS:
    eligible = [row for row in records if row.get('cer_eligible') == 'yes' and row.get('transcription_verified', '').strip()]
    print(f'{set_name}: {len(eligible)} labelled pairs')
    for number, row in enumerate(eligible, start=1):
        raw_bgr = cv2.imread(str(PAIR_REPO / row['input_path']))
        target_bgr = cv2.imread(str(PAIR_REPO / row['target_path']))
        background = pair_data.board_background(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
        inp, tgt, ok = pair_trainer.load_pair(row)                              # normalised grey
        restored = pair_trainer.restore_full(test_model, inp, eval_device)      # native scale, as trained
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        no_cnn_grey = pair_data.denormalise_grey(inp, background)             # back to natural brightness
        restored_grey = pair_data.denormalise_grey(restored, background)
        images = {
            'raw input': raw_bgr,
            'input, no CNN': no_cnn_grey,
            'restored': restored_grey,
            'input, no CNN, colour': cv2.cvtColor(pair_data.put_colour_back(no_cnn_grey, raw_rgb), cv2.COLOR_RGB2BGR),
            'restored, colour': cv2.cvtColor(pair_data.put_colour_back(restored_grey, raw_rgb), cv2.COLOR_RGB2BGR),
            'clear target': target_bgr,
        }
        record = {'set': set_name, 'pair_id': row['pair_id'], 'input_role': row['input_role'],
                  'spp_band': pair_data.spp_band(row['source_pixels_per_target_pixel']),
                  'ground_truth': row['transcription_verified']}
        for version in VERSIONS:
            text, confidence = recognise_line(shrink_for_ocr(images[version]))
            record[f'{version} text'] = text
            record[f'{version} CER'] = character_error_rate(row['transcription_verified'], text)
        ocr_rows.append(record)
        print(f"  {number:>2}/{len(eligible)} CER raw {record['raw input CER']:.3f} | grey: no CNN {record['input, no CNN CER']:.3f} "
              f"restored {record['restored CER']:.3f} | colour: no CNN {record['input, no CNN, colour CER']:.3f} "
              f"restored {record['restored, colour CER']:.3f} | target {record['clear target CER']:.3f}")

ocr_comparison = pd.DataFrame(ocr_rows)
ocr_path = PAIR_REPO / 'outputs' / 'paired_dataset_17oct' / f'{RUN_TAG}_ocr_comparison.csv'
ocr_comparison.to_csv(ocr_path, index=False)

summary = ocr_comparison.groupby('set')[[f'{v} CER' for v in VERSIONS]].mean().round(3)

def counts(before, after):
    # improved / unchanged / worsened per set when going from version `before` to version `after`
    gain = ocr_comparison[f'{before} CER'] - ocr_comparison[f'{after} CER']
    return [f"{int((g > 0).sum())} / {int((g == 0).sum())} / {int((g < 0).sum())}"
            for _, g in gain.groupby(ocr_comparison['set'])]

summary['colour: restored vs no CNN (better / same / worse)'] = counts('input, no CNN, colour', 'restored, colour')
summary['colour: restored vs raw'] = counts('raw input', 'restored, colour')
summary['grey: restored vs no CNN'] = counts('input, no CNN', 'restored')
print(f'\nPaddleOCR CER (lower is better) - run {RUN_TAG}')
display(summary)
display(ocr_comparison.groupby(['set', 'spp_band'])[[f'{v} CER' for v in VERSIONS]].mean().round(3))
print(f'Saved detailed comparison: {ocr_path}')

# Pass/fail rule (fixed before M1 was trained): on REAL pairs from an UNSEEN board
REAL_UNSEEN_SETS = ['Airdrop, unseen board']     # add the far-shot photo set here once it exists
PATHS = [('colour path (primary)', 'restored, colour', 'input, no CNN, colour'),
         ('grey path (reported)', 'restored', 'input, no CNN')]
for set_name in REAL_UNSEEN_SETS:
    part = ocr_comparison[ocr_comparison['set'] == set_name]
    lines = part['ground_truth'].nunique()
    print(f"\n{set_name} ({len(part)} pairs, {lines} different lines - preliminary until the far-shot set exists)")
    for path_name, restored_v, no_cnn_v in PATHS:
        restored, no_cnn, raw = (part[f'{v} CER'].mean() for v in (restored_v, no_cnn_v, 'raw input'))
        diff = part[f'{no_cnn_v} CER'] - part[f'{restored_v} CER']
        checks = {f'restored CER {restored:.3f} < no CNN {no_cnn:.3f}': restored < no_cnn,
                  f'restored CER {restored:.3f} < raw input {raw:.3f}': restored < raw,
                  f'improved {int((diff > 0).sum())} > worsened {int((diff < 0).sum())}': (diff > 0).sum() > (diff < 0).sum()}
        print(f"  {path_name}: {'PASS' if all(checks.values()) else 'FAIL'}")
        for text, ok in checks.items():
            print(f"    {'yes' if ok else 'no '}  {text}")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


17 Oct, held-out photos: 37 labelled pairs
   1/37 CER raw 0.476 | grey: no CNN 0.333 restored 0.333 | colour: no CNN 0.333 restored 0.333 | target 0.286
   2/37 CER raw 0.125 | grey: no CNN 0.083 restored 0.125 | colour: no CNN 0.125 restored 0.125 | target 0.167
   3/37 CER raw 0.238 | grey: no CNN 0.143 restored 0.190 | colour: no CNN 0.143 restored 0.190 | target 0.190
   4/37 CER raw 1.000 | grey: no CNN 1.000 restored 1.000 | colour: no CNN 1.000 restored 1.000 | target 1.000
   5/37 CER raw 0.167 | grey: no CNN 0.042 restored 0.000 | colour: no CNN 0.083 restored 0.083 | target 0.042
   6/37 CER raw 0.077 | grey: no CNN 0.077 restored 0.077 | colour: no CNN 0.038 restored 0.038 | target 0.038
   7/37 CER raw 0.037 | grey: no CNN 0.037 restored 0.037 | colour: no CNN 0.037 restored 0.037 | target 0.037
   8/37 CER raw 0.000 | grey: no CNN 0.000 restored 0.000 | colour: no CNN 0.000 restored 0.000 | target 0.000
   9/37 CER raw 0.226 | grey: no CNN 0.161 restored 0.161 | colour: n

,raw input CER,"input, no CNN CER",restored CER,"input, no CNN, colour CER","restored, colour CER",clear target CER,colour: restored vs no CNN (better / same / worse),colour: restored vs raw,grey: restored vs no CNN
set,,,,,,,,,
"17 Oct, held-out photos",0.116,0.095,0.087,0.092,0.090,0.099,5 / 29 / 3,16 / 19 / 2,11 / 21 / 5
"Airdrop, unseen board",0.157,0.169,0.181,0.165,0.188,0.067,0 / 15 / 9,1 / 12 / 11,3 / 13 / 8
"PhotosToUse, camera shake (stress test)",0.570,0.556,0.596,0.549,0.587,0.051,2 / 2 / 4,0 / 6 / 2,1 / 4 / 3
"PhotosToUse, low resolution (synthetic)",0.153,0.184,0.202,0.172,0.225,0.037,0 / 20 / 10,2 / 14 / 14,3 / 19 / 8


raw input CER  \
set                                     spp_band                              
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)          0.065   
                                        <0.15 (>6.7x)                 0.214   
                                        >=0.30 (<3.3x)                0.161   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)          0.197   
                                        >=0.30 (<3.3x)                0.146   
PhotosToUse, camera shake (stress test) >=0.30 (<3.3x)                0.570   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)          0.238   
                                        >=0.30 (<3.3x)                0.068   

                                                              input, no CNN CER  \
set                                     spp_band                                  
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)              0.051   
                                        <0.15 (>6.7x)                     0.143   
                                        >=0.30 (<3.3x)                    0.136   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)              0.228   
                                        >=0.30 (<3.3x)                    0.154   
PhotosToUse, camera shake (stress test) >=0.30 (<3.3x)                    0.556   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)              0.310   
                                        >=0.30 (<3.3x)                    0.059   

                                                              restored CER  \
set                                     spp_band                             
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)         0.041   
                                        <0.15 (>6.7x)                0.143   
                                        >=0.30 (<3.3x)               0.130   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)         0.251   
                                        >=0.30 (<3.3x)               0.163   
PhotosToUse, camera shake (stress test) >=0.30 (<3.3x)               0.596   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)         0.327   
                                        >=0.30 (<3.3x)               0.077   

                                                              input, no CNN, colour CER  \
set                                     spp_band                                          
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)                      0.045   
                                        <0.15 (>6.7x)                             0.143   
                                        >=0.30 (<3.3x)                            0.136   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)                      0.215   
                                        >=0.30 (<3.3x)                            0.152   
PhotosToUse, camera shake (stress test) >=0.30 (<3.3x)                            0.549   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)                      0.274   
                                        >=0.30 (<3.3x)                            0.070   

                                                              restored, colour CER  \
set                                     spp_band                                     
17 Oct, held-out photos                 0.15-0.30 (3.3-6.7x)                 0.046   
                                        <0.15 (>6.7x)                        0.143   
                                        >=0.30 (<3.3x)                       0.131   
Airdrop, unseen board                   0.15-0.30 (3.3-6.7x)                 0.264   
                                        >=0.30 (<3.3x)                       0.168   
PhotosToUse, camera shake (stress test) >=0.30 (<3.3x)                       0.587   
PhotosToUse, low resolution (synthetic) 0.15-0.30 (3.3-6.7x)

Saved detailed comparison: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/paired_dataset_17oct/m2_recognition_loss_ocr_comparison.csv

Airdrop, unseen board (24 pairs, 6 different lines - preliminary until the far-shot set exists)
  colour path (primary): FAIL
    no   restored CER 0.188 < no CNN 0.165
    no   restored CER 0.188 < raw input 0.157
    no   improved 0 > worsened 9
  grey path (reported): FAIL
    no   restored CER 0.181 < no CNN 0.169
    no   restored CER 0.181 < raw input 0.157
    no   improved 3 > worsened 8


### 6d. Camera-shake check: retake instead of restore

Restoration cannot recover text smeared by camera shake, but a retake can. `scripts/capture_check.py` measures each line crop's **sharpness** and how strongly its edges share **one direction** (shake smears every stroke along the shake direction; a far or soft photo is blurred evenly). Thresholds were calibrated on the 11 real shake crops against 1,014 other line crops (Airdrop, 17 Oct, PhotosToUse clear variants): all 11 caught, 10/11 when each is left out of the fit, and the one other crop flagged (`IMG_4505_L02`, "apple car") turned out to be visibly shaken too.

This cell runs the check on every test input. Expected: the camera-shake set is flagged; the low-resolution set (soft but not directional) is not. The flagged pairs' CER shows why the app should ask for a retake rather than restore.

In [35]:
import importlib.util
import cv2
import pandas as pd
cc_spec = importlib.util.spec_from_file_location('capture_check', PAIR_REPO / 'scripts' / 'capture_check.py')
capture_check = importlib.util.module_from_spec(cc_spec)
cc_spec.loader.exec_module(capture_check)
thresholds = capture_check.load_thresholds()
print(f"rule: camera shake if coherence >= {thresholds['coherence_min']} and sharpness <= {thresholds['sharpness_max']}"
      f" | calibrated on {thresholds['calibrated_on']}")

check_rows = []
for set_name, records in TEST_SETS:
    for row in records:
        image = cv2.imread(str(PAIR_REPO / row['input_path']))
        check_rows.append({'set': set_name, 'pair_id': row['pair_id'], **capture_check.features(image),
                           'advice': capture_check.capture_advice(image, thresholds=thresholds)})
capture_results = pd.DataFrame(check_rows)
display(capture_results.groupby(['set', 'advice']).size().unstack(fill_value=0))
if 'ocr_comparison' in globals():
    merged = ocr_comparison.merge(capture_results[['pair_id', 'advice']], on='pair_id')
    display(merged.groupby(['set', 'advice'])[['raw input CER', 'restored CER', 'clear target CER']].mean().round(3))

rule: camera shake if coherence >= 0.6348 and sharpness <= 0.01608 | calibrated on {'17 Oct line crops': 392, 'Airdrop line crops': 601, 'PhotosToUse clear variants': 21, 'shake (real)': 11}


advice,ok,retake: camera shake
set,,
"17 Oct, held-out photos",40,0
"Airdrop, unseen board",24,0
"PhotosToUse, camera shake (stress test)",0,8
"PhotosToUse, low resolution (synthetic)",30,0


,,raw input CER,restored CER,clear target CER
set,advice,,,
"17 Oct, held-out photos",ok,0.116,0.087,0.099
"Airdrop, unseen board",ok,0.157,0.181,0.067
"PhotosToUse, camera shake (stress test)",retake: camera shake,0.570,0.596,0.051
"PhotosToUse, low resolution (synthetic)",ok,0.153,0.202,0.037


### 6e. Multi-photo reading fusion: recover what one photo lost

Every SR model above restores a line from ONE photo, and a far photo simply does not contain the lost strokes. But a line is often photographed more than once, and a stroke that is lost in one photo can survive in another: on the CNN board, picking the best photo of each line would lower CER from about 0.16 to about 0.09. This cell fuses PaddleOCR's **readings** of the same line across photos (no pixel alignment needed; in the app, lines are matched across photos by box overlap after rectification, W8 IoU).

Rules (`scripts/reading_fusion.py`, after Nascimento et al. 2025, who fused the OCR outputs of 5 frames per licence plate):
- **hc**: the reading PaddleOCR is most confident about;
- **mv**: the most frequent reading;
- **medoid**: the reading closest to all others;
- **vote**: every reading is aligned to the medoid by edit distance, then each character (and each inserted character) is decided by a confidence-weighted vote (ROVER);
- **oracle**: the best single photo (needs the truth: an upper bound, not a usable rule).

**How to read the table.** Each row uses only lines with at least *k* photos (k = all: at least 2), and every column is scored on those same lines. `single` is the expected CER of ONE photo, which is today's pipeline. A rule helps if it is clearly below `single` (by more than about 0.02, the noise level found earlier). Rows are given for the raw photo and for the SR model of `RUN_TAG` (with colour), to see whether SR + fusion beats fusion alone. Sample sizes are small (CNN board: 5 lines with several photos; 17 Oct: 16 lines with 2 photos each), so treat the result as preliminary.

In [ ]:
import importlib.util
from collections import defaultdict

import cv2
import pandas as pd

fspec = importlib.util.spec_from_file_location('reading_fusion', PAIR_REPO / 'scripts' / 'reading_fusion.py')
reading_fusion = importlib.util.module_from_spec(fspec)
fspec.loader.exec_module(reading_fusion)

if 'accepted_recogniser' not in globals():
    from paddleocr import TextRecognition
    accepted_recogniser = TextRecognition(model_name='PP-OCRv6_medium_rec')

def read_with_confidence(image_bgr, max_side=1600):
    # same shrink as 6c, then PaddleOCR's text AND its confidence
    h, w = image_bgr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        image_bgr = cv2.resize(image_bgr, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    result = accepted_recogniser.predict([image_bgr])[0]
    return str(result['rec_text']), float(result['rec_score'])

FUSION_SETS = ['Airdrop, unseen board', '17 Oct, held-out photos']   # sets where a line was photographed more than once
reading_rows = []
for set_name, records in TEST_SETS:
    if set_name not in FUSION_SETS:
        continue
    eligible = [row for row in records if row.get('cer_eligible') == 'yes' and row.get('transcription_verified', '').strip()]
    print(f'{set_name}: reading {len(eligible)} photos (raw and SR) ...')
    for row in eligible:
        raw_bgr = cv2.imread(str(PAIR_REPO / row['input_path']))
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        inp, _, _ = pair_trainer.load_pair(row)
        restored = pair_trainer.restore_full(test_model, inp, eval_device)
        colour = pair_data.put_colour_back(pair_data.denormalise_grey(restored, pair_data.board_background(raw_rgb)), raw_rgb)
        raw_text, raw_conf = read_with_confidence(raw_bgr)
        sr_text, sr_conf = read_with_confidence(cv2.cvtColor(colour, cv2.COLOR_RGB2BGR))
        reading_rows.append({'set': set_name, 'line': row['transcription_verified'], 'pair_id': row['pair_id'],
                             'photo': row['input_photo'], 'raw text': raw_text, 'raw conf': raw_conf,
                             'sr text': sr_text, 'sr conf': sr_conf})
readings = pd.DataFrame(reading_rows)
readings_path = PAIR_REPO / 'outputs' / 'paired_dataset_17oct' / f'{RUN_TAG}_readings_with_confidence.csv'
readings.to_csv(readings_path, index=False)

rows = []
for set_name in FUSION_SETS:
    part = readings[readings['set'] == set_name]
    for version, text_col, conf_col in [('raw photo', 'raw text', 'raw conf'), (f'SR {RUN_TAG} (colour)', 'sr text', 'sr conf')]:
        groups = {line: (line, list(zip(g[text_col], g[conf_col]))) for line, g in part.groupby('line')}
        for result in reading_fusion.evaluate(groups):
            rows.append({'set': set_name, 'version': version, **result})
fusion_results = pd.DataFrame(rows)
fusion_table = fusion_results.pivot_table(index=['set', 'version', 'photos per line', 'lines'], columns='rule',
                                          values='mean CER')[['single', 'hc', 'mv', 'medoid', 'vote', 'oracle']].round(3)
print('Mean CER per line (lower is better); every column in a row uses the same lines')
display(fusion_table)
print(f'Saved readings with confidence: {readings_path}')